# Phase 6 — Abstractive Legal Summarization (CS781 Capstone) — Kaggle

Starts **every** experiment from a **Phase-1 Legal Language Model**  and follows the Phase 6 spec .

| Spec requirement | Where it is handled |
|---|---|
| Data: `Train/train.json` (id, document, summary), `Test/test.json`; **no dev split** → hold-out split from train | §3 |
| All **four** Phase-1 architectures, **best four submissions on four different models** | §6 grid, §10 selection |
| ≥ 4 substantially different experiments, ≥ 3 different adaptation techniques, **LoRMA compulsory** | LoRMA-π (+ LoRMA-+), LoRA, Pfeiffer / Houlsby adapters, prompt tuning, full fine-tuning |
| **Knowledge distillation** (larger/stronger teacher → smaller student, counts as one of the 4) | §7 |
| Long documents: chunking, overlap, hierarchical, importance selection; *is legal information lost?* | §4, §8 |
| Metrics: BERTScore F1 (primary), ROUGE-1 F1, ROUGE-L F1 | every table |
| Output: list-format JSON `[{"id": ..., "summary": ...}]`, exactly one summary per test document | §10 |
| Analysis (§12.5): technique effectiveness, PEFT vs full, long-doc strategies, factuality / coverage / redundancy, length, domain adaptation, KD, size/memory/cost, usefulness of Phase-1 representations | §11 |

## How it is organised (same style as Phase 1)
Code lives in four files written by §2 (`p6text.py` pure-python text tools · `p6lib.py` model surgery / PEFT / decoding / metrics · `run_exp.py` train + validate one experiment · `infer.py` validation scoring and test submission). Each experiment runs as its **own subprocess** via `run()`, so an out-of-memory kill or a crash stops the notebook loudly instead of silently, and finished experiments are skipped on re-run (`artifacts/exps/<name>.pt`). Every result is appended to `artifacts/results.jsonl`.

## Design decisions worth knowing (put them in the report)
* **One summarisation interface for all four architectures.** Decoder-style models (`decoder_only`, `efficient`, `chunk_recurrent`) see `[bos] document <\n\nSummary:\n> summary [eos]` and are trained **only on the summary tokens**; `enc_dec` encodes the document and decodes the summary. The same `forward_pass()` returns *summary-position logits only*, which also makes cross-architecture distillation possible.
* **PEFT is implemented from scratch** (no `peft` library; LoRMA is not in it anyway): LoRA, **LoRMA** (multiplicative `W' = I_π(BA)·W₀`, permutation rank-inflation + the paper's identity initialisation; additive-inflation variant too), Houlsby / Pfeiffer bottleneck adapters, prompt tuning, and full fine-tuning. `nn.MultiheadAttention` packs q/k/v and bypasses `out_proj.forward`, so `enc_dec` / `chunk_recurrent` attention is first converted to an equivalent module with separate Linear layers (`SplitMHA`; §5 *proves* the logits are unchanged). LoRA/LoRMA are merged into the weights before inference (zero extra latency).
* **Distillation.** Teacher = the strongest fine-tuned model so far; student = a **depth-pruned copy (every 2nd layer) of a Phase-1 model**, so it still starts from Phase-1 weights. Loss = `α·CE + (1-α)·T²·KL(teacher‖student)` on summary tokens, plus a *same student without teacher* control that isolates the effect of distillation.
* **Controls (analysis only, never submitted):** a randomly-initialised full-fine-tuning run (does Phase-1 pre-training help?) and the pruned-student-without-teacher run.
* **Long documents.** Training input = importance-selected sentences (`select`: TF-IDF salience × position prior × legal cue words, MMR-style redundancy filter) by default; inference can also use `trunc`, `headtail`, or **hierarchical** chunk-summaries → document summary. §4 measures, model-free, how much of the reference summary's information is even *visible* under each strategy.
* **Decoding:** my own beam search (works for all four architectures) with repetition penalty, n-gram blocking, length penalty, min/max length and an optional **copy bias** (soft logit bonus for tokens of the selected source text).

**Prerequisites on Kaggle:** Internet On; a GPU (T4 ×2 / P100 / A100); a dataset containing your **Phase-1 outputs** (`tokenizer.json`, the `*_best.pt` checkpoints of all four architectures, and `models.py` + `config.py` from Phase 1) and the **Phase 6 data** (`Phase_6_Summ/Train/train.json`, `Test/test.json`). Everything is auto-discovered; override paths in §0 if needed.

## What changed in this version (v3 = v2 + rows 7-8) and why

The first full run (decoder_only / enc_dec / efficient, 50 validation documents) gave BERTScore F1 0.79-0.82; LoRA and Pfeiffer led, **LoRMA was last on every architecture**. Changes aimed at the primary metric:

| # | Change | Why | Cost |
|---|---|---|---|
| 1 | `lorma_pi` learning rate **5e-5 → 2e-4**, plus a `6e-4` bracket run on `decoder_only` | LoRMA's validation loss was still falling at step 600 and was far above LoRA's (3.98 vs 3.81); with Adam each parameter moves ≈ lr per step, so 5e-5 barely moves the near-identity matrix | +1 run (optional) |
| 2 | Training length **600 → 800 steps** (≈ 2 epochs) | every method's validation loss was still improving; PEFT methods learn slowly | ≈ +2-3 min per run |
| 3 | **Length sweep fixed** (§9) | the old sweep used fractions of the median reference length capped at `MAX_SUM`, so 4 of its 5 length settings were identical; now 0.4 / 0.6 / 0.8 / 1.0 × `MAX_SUM` | none |
| 4 | **Decoding sweep widened** (§9): repetition penalty 1.0 / 1.05 (v1 default 1.1 penalises ordinary words such as "the", "of"), n-gram blocking 3 / 4 / 6 | the sweep picks `FINAL_GEN` for the submissions; v1 only tried 1.3 on the penalty and 3 on blocking | same number of settings |
| 5 | **Time guard** (§0, §6) | the full pipeline is ≈ 10 h; the optional runs (Houlsby, prompt tuning, LoRMA-+, LoRMA lr bracket) are now last and are skipped once the session passes 7 h, so the decoding sweep and the four test submissions can never be cut off by Kaggle's 12 h limit | none |
| 6 | `full_*` and the random-init control now run **before** the optional extras | they matter for the best-model selection and for §12.5 | none |

| 7 | **Copy-biased decoding** (§9, stage 2): logit bonus 1 / 2 / 3 for tokens that occur in the selected source text, tried on top of the best length / repetition setting | the models are small and trained from scratch (validation loss ≈ 3.8); references reuse the document's names, sections and terms, so nudging generation toward source tokens raises lexical and semantic overlap. Still generation by the fine-tuned model | +3 sweep settings (≈ 15 min) |
| 8 | **`enc_dec` writes longer summaries**: cap 400 → 640 tokens (`ENC_DEC_MAX_SUM`), generation length scaled automatically | references have a median of 773 tokens, so 400 caps recall; `enc_dec` has a separate decoder, so a longer summary costs it no document context (decoder-style models share one window, so they stay at 400) | enc_dec runs ≈ 2x slower (training +40 %, decoding ≈ 2-3x) |

Everything else (models, PEFT code, KD, long-document strategies, analysis cells) is unchanged. Not changed on purpose: beam-search has no KV cache (decoding is ≈ 13 s/document for decoder-style models and dominates the run time) because that needs edits inside the Phase-1 `models.py` that cannot be checked here.


## 0. Settings + dependencies

In [ ]:
import time
NB_START = time.time()          # session clock for the time guard (see have_time() in section 6)
SESSION_LIMIT_H = 11.5          # Kaggle sessions are killed at 12 h; stay under it
POST_GRID_RESERVE_H = 4.5       # hours kept free for KD + long-doc strategies + decoding sweep + 4 test submissions (~4 h)
ELAPSED_OFFSET_H = 0.0          # re-opening a session that has already been running? put the hours already used here

QUICK = False            # True -> tiny budgets, whole notebook in ~30 min (smoke test of every cell)
FORCE = False            # True -> retrain experiments even if artifacts/exps/<name>.pt already exists (overwrites!)
SMOKE = True             # run the ~5-10 min smoke test in section 5 before the long runs
SEED  = 42
ENC_DEC_MAX_SUM = 640   # enc_dec only: its document window does not shrink when summaries get longer (set None to disable)
CTX   = 1024             # context for every model (RoPE models are extended to it; enc_dec position table is grown)
TRAIN_STRATEGY = "select"  # long-document input used for TRAINING: trunc | headtail | select
USE_BERT = True          # BERTScore needs internet (downloads roberta-large once)
BERT_MODEL = None        # None = bert-score default for English; "distilroberta-base" is a faster stand-in for development
ARCHS = ["decoder_only", "enc_dec", "efficient", "chunk_recurrent"]

# Optional manual paths (None / {} = auto-discover under /kaggle/input and the working directory)
P1_CKPT_OVERRIDE = {}    # e.g. {"decoder_only": "/kaggle/input/p1/ckpt_tier2/decoder_only_best.pt", "enc_dec": ..., ...}
TOKENIZER_OVERRIDE = None
TRAIN_OVERRIDE = TEST_OVERRIDE = None

if QUICK:
    S = dict(val_n=24, eval_n=6,  steps=30,  eval_every=15, warmup=5,  bs=2, accum=1, max_train_docs=80, val_loss_n=8,
             strat_n=6, hier_n=3, sweep_n=4)
else:
    # ~18-28 min per run on a T4; 800 steps = ~2 epochs of the 6.8k training documents (v1 used 600 = 1.4 epochs and was still improving)
    S = dict(val_n=200, eval_n=50, steps=800, eval_every=100, warmup=40, bs=4, accum=4, max_train_docs=0, val_loss_n=48,
             strat_n=40, hier_n=12, sweep_n=30)
print("QUICK" if QUICK else "FULL", S)

In [ ]:
# Internet must be On.  torch / tokenizers are preinstalled on Kaggle.
!pip install -q bert-score rouge-score matplotlib

In [ ]:
import importlib, os
for m in ["torch", "tokenizers", "rouge_score", "bert_score", "pandas", "numpy", "matplotlib"]:
    try:
        mod = importlib.import_module(m); print(f"{m:12s} {getattr(mod, '__version__', 'ok')}")
    except Exception as e:
        print(f"{m:12s} FAILED: {type(e).__name__}: {e}")
import torch
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory/1e9:.0f} GB, capability {p.major}.{p.minor} -> autocast {'bf16' if p.major >= 8 else 'fp16'}")
else:
    print("WARNING: no GPU -- training and beam search will be extremely slow")

## 1. Locate the data, the Phase-1 models and the Phase-1 code

Searches `/kaggle/input` (and the working directory) for: `train.json` / `test.json` of Phase 6, `tokenizer.json`, `<arch>_best.pt` for the four architectures (a `ckpt_tier2` copy is preferred over `ckpt_tier1`), and Phase-1 `models.py` / `config.py` (needed to rebuild the architectures; copied to `phase1_code/`).

**Context note.** Tier-1 checkpoints were trained at context 512. They work (RoPE models are simply extended to `CTX` and fine-tuned at that length; the `enc_dec` position table is grown), but a Tier-2 checkpoint with a 1024 context is better. Use the strongest checkpoint you have for every architecture and say which in the report.

In [ ]:
import os, sys, json, shutil, shlex, time, random
ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else os.getcwd()
os.chdir(WORK); os.makedirs("artifacts", exist_ok=True)
BASES = [b for b in ["/kaggle/input", WORK] if os.path.isdir(b)]

def find_all(filename):
    out = []
    for b in BASES:
        for root, _, files in os.walk(b):
            if filename in files:
                out.append(os.path.join(root, filename))
    return out

# ---- Phase 6 data
def pick(cands, prefer):
    cands = sorted(cands, key=lambda p: (0 if any(k in p.lower() for k in prefer) else 1, len(p)))
    return cands[0] if cands else None
TRAIN_FILE = TRAIN_OVERRIDE or pick([p for p in find_all("train.json") if "phase_1" not in p.lower()], ["summ", "phase_6", "phase6"])
TEST_FILE  = TEST_OVERRIDE  or pick([p for p in find_all("test.json") if "phase_1" not in p.lower()],  ["summ", "phase_6", "phase6"])
assert TRAIN_FILE and TEST_FILE, "Phase 6 train.json / test.json not found under /kaggle/input -- add the dataset or set TRAIN_OVERRIDE / TEST_OVERRIDE"
print("train:", TRAIN_FILE, f"({os.path.getsize(TRAIN_FILE)/1e6:.1f} MB)\ntest :", TEST_FILE, f"({os.path.getsize(TEST_FILE)/1e6:.1f} MB)")

# ---- Phase 1 tokenizer
TOKENIZER = TOKENIZER_OVERRIDE or pick(find_all("tokenizer.json"), ["artifacts"])
assert TOKENIZER, "Phase-1 tokenizer.json not found -- add your Phase-1 outputs as a dataset or set TOKENIZER_OVERRIDE"
print("tokenizer:", TOKENIZER)

# ---- Phase 1 checkpoints (all four architectures)
P1 = {}
for arch in ARCHS:
    c = P1_CKPT_OVERRIDE.get(arch) or pick(find_all(f"{arch}_best.pt"), ["tier2", "tier1"])
    assert c, f"Phase-1 checkpoint {arch}_best.pt not found -- all four architectures are required (spec 12.1). Set P1_CKPT_OVERRIDE."
    P1[arch] = c
print("phase-1 checkpoints:"); [print(f"  {a:16s} {p}") for a, p in P1.items()]

# ---- Phase 1 code (architectures)
def find_code(name, marker):
    c = [p for p in find_all(name) if marker in open(p, errors="ignore").read()]
    return c[0] if c else None
m_py, c_py = find_code("models.py", "ChunkRecurrentLM"), find_code("config.py", "class ModelConfig")
assert m_py and c_py, "Phase-1 models.py / config.py not found -- add them to the Phase-1 dataset (or upload next to this notebook)"
os.makedirs("phase1_code", exist_ok=True)
for src in (m_py, c_py):
    dst = os.path.join("phase1_code", os.path.basename(src))
    if os.path.abspath(src) != os.path.abspath(dst):
        shutil.copy(src, dst)
print("phase-1 code:", m_py, c_py, "-> phase1_code/")

## 2. Write project files

`p6text.py` (sentence splitting, importance selection, overlapping chunks, redundancy/coverage/fact metrics — no torch) · `p6lib.py` (SplitMHA, depth pruning, LoRA / **LoRMA** / adapters / prompt tuning, budgets, beam search, hierarchical summariser, metrics) · `run_exp.py` (train + validate one experiment, incl. distillation) · `infer.py` (validation scoring, test submission).

In [ ]:
%%writefile p6text.py
"""
Pure-python text utilities for Phase 6 (NO torch needed, so they can be unit-tested anywhere).

  * sentence splitting, importance-based sentence selection, sentence-aligned overlapping chunking
  * redundancy / extractiveness / coverage / "legal-fact" metrics used in the analysis section
"""
import math
import random
import re
from collections import Counter

STOP = set("""a an the of to in and or for on at by with from as is are was were be been being this that these those it its
which who whom whose he she they them his her their there here not no nor but if then than so such into over under after
before between about against upon any all each other more most some can could may might shall should will would also
has have had do does did i we you our us your however thus hence therefore whereas where when what while whether""".split())

CUES = ("held that", "we hold", "it is held", "hereby", "directed", "dismissed", "allowed", "set aside", "appeal",
        "petition", "respondent", "appellant", "accused", "convicted", "acquitted", "section", "article", "act,",
        "judgment", "order", "decree", "sentence", "bail", "writ", "contract", "tribunal", "accordingly",
        "in the result", "for the reasons", "conclusion", "in view of", "we are of the opinion", "the question")

_SENT = re.compile(r'(?<=[\.\?\!;])\s+(?=[A-Z0-9\(\"\'“])|\n\s*\n+|\n(?=\s*(?:\d+[\.\)]|\([a-zA-Z0-9]{1,4}\)))')
_WORD = re.compile(r"[A-Za-z0-9][A-Za-z0-9'\-]*")
_NUM = re.compile(r"\b\d[\d,\.]*\d\b|\b\d\b")
_SEC = re.compile(r"\b(?:section|sections|article|articles|rule|rules|order|clause)\s+(\d+[A-Za-z]?)", re.I)
_CAP = re.compile(r"\b[A-Z][a-z]{2,}\b")


def words(text):
    return _WORD.findall(text.lower())


def content_set(text):
    return {w for w in words(text) if w not in STOP and len(w) > 2}


def split_sentences(text):
    parts = [p.strip() for p in _SENT.split(text) if p and p.strip()]
    return parts or ([text.strip()] if text.strip() else [])


# ---------------------------------------------------------------------------
# Importance-based selection (unsupervised: TF-IDF salience x position prior + legal cue bonus, MMR-lite)
# ---------------------------------------------------------------------------
def score_sentences(sents):
    n = len(sents)
    toks = [[w for w in words(s) if w not in STOP and len(w) > 2] for s in sents]
    tf = Counter(w for t in toks for w in t)
    sdf = Counter(w for t in toks for w in set(t))
    wt = {w: (1.0 + math.log(tf[w])) * math.log(1.0 + n / sdf[w]) for w in tf}
    sal = [sum(wt[w] for w in set(t)) / (math.sqrt(len(t)) + 1.0) for t in toks]
    mx = max(sal) if sal and max(sal) > 0 else 1.0
    out = []
    for i, s in enumerate(sents):
        pos = i / max(1, n - 1)
        prior = 1.0 + (0.5 if pos < 0.12 else 0.0) + (0.6 if pos > 0.88 else 0.0)
        low = s.lower()
        cue = 0.15 * min(3, sum(1 for c in CUES if c in low))
        out.append(sal[i] / mx * prior + cue)
    return out


def select_sentences(sents, budget, lens, redundancy_thr=0.7):
    """Greedy highest-score-first selection under a token budget; returns indices in ORIGINAL order."""
    if sum(lens) <= budget:
        return list(range(len(sents)))
    scores = score_sentences(sents)
    order = sorted(range(len(sents)), key=lambda i: -scores[i])
    chosen, used, chosen_sets = [], 0, []
    for i in order:
        if used + lens[i] > budget:
            continue
        cs = content_set(sents[i])
        if cs and any(len(cs & o) / max(1, len(cs | o)) > redundancy_thr for o in chosen_sets):
            continue
        chosen.append(i); chosen_sets.append(cs); used += lens[i]
        if budget - used < 8:
            break
    return sorted(chosen)


def chunk_sentences(lens, budget, overlap=1):
    """Sentence-aligned chunks of <= budget tokens, consecutive chunks share `overlap` sentences."""
    n, i, chunks = len(lens), 0, []
    while i < n:
        j, tot = i, 0
        while j < n and tot + lens[j] <= budget:
            tot += lens[j]; j += 1
        if j == i:
            j = i + 1                       # a single over-long sentence: caller truncates it
        chunks.append((i, j))
        if j >= n:
            break
        k = max(i + 1, j - overlap)
        # only overlap if the next chunk can still advance past j; otherwise start right after this chunk
        i = k if sum(lens[k:j + 1]) <= budget else j
    return chunks


def make_split(ids, val_n, seed=42):
    ids = list(ids)
    random.Random(seed).shuffle(ids)
    return {"val": ids[:val_n], "train": ids[val_n:]}


# ---------------------------------------------------------------------------
# Analysis metrics
# ---------------------------------------------------------------------------
def facts(text):
    """Cheap proxy for 'legal facts': numbers, statute/section references and capitalised names."""
    f = {"n:" + m.replace(",", "") for m in _NUM.findall(text)}
    f |= {"s:" + m.lower() for m in _SEC.findall(text)}
    f |= {"c:" + m.lower() for m in _CAP.findall(text) if m.lower() not in STOP}
    return f


def recall(ref_set, cand_set):
    return len(ref_set & cand_set) / len(ref_set) if ref_set else float("nan")


def ngrams(toks, n):
    return [tuple(toks[i:i + n]) for i in range(len(toks) - n + 1)]


def repeat_rate(text, n=3):
    g = ngrams(words(text), n)
    return (len(g) - len(set(g))) / len(g) if g else 0.0


def distinct_n(text, n=2):
    g = ngrams(words(text), n)
    return len(set(g)) / len(g) if g else 1.0


def extractiveness(summary, source, n=4):
    g = ngrams(words(summary), n)
    if not g:
        return float("nan")
    src = set(ngrams(words(source), n))
    return sum(1 for x in g if x in src) / len(g)


def analyse(pred, ref, src):
    """One row of per-document analysis numbers (all in [0, 1] except `words`)."""
    pf, rf, sf = facts(pred), facts(ref), facts(src)
    pc, rc, sc = content_set(pred), content_set(ref), content_set(src)
    return {
        "words": len(pred.split()),
        "rep3": repeat_rate(pred, 3),
        "distinct2": distinct_n(pred, 2),
        "ext4": extractiveness(pred, src, 4),
        "ref_fact_recall": recall(rf, pf),
        "ref_content_recall": recall(rc, pc),
        "fact_hallucination": (len(pf - sf) / len(pf)) if pf else float("nan"),   # facts in summary absent from source
        "ref_fact_in_src": recall(rf, sf),                                         # ceiling: how much of the ref is even present
        "ref_content_in_src": recall(rc, sc),
    }

In [ ]:
%%writefile p6lib.py
"""
Phase 6 library: model surgery (MHA split, depth pruning, context extension), PEFT methods implemented from
scratch (LoRA, LoRMA, Houlsby / Pfeiffer adapters, prompt tuning), data / budget handling, beam-search decoding,
hierarchical summarisation and metrics.  Works on top of the Phase-1 models.py / config.py (copied by the notebook
into ./phase1_code).
"""
import contextlib
import copy
import json
import math
import os
import random
import sys
import time

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "phase1_code"))

import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

import p6text as T
from models import build_model, load_checkpoint, get_amp_dtype

PAD, BOS, EOS, UNK = 0, 1, 2, 3          # Phase-1 tokenizer special ids
SEP_TEXT = "\n\nSummary:\n"
ENC_SRC_MAX = 768
RESULTS = "artifacts/results.jsonl"
EXP_DIR = "artifacts/exps"
ATTN_T = ("q_proj", "k_proj", "v_proj", "o_proj")
FFN_T = ("w1", "w2", "w3")


def family(arch):
    return "encdec" if arch == "enc_dec" else "causal"


def set_seed(s):
    random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def append_result(row):
    os.makedirs(os.path.dirname(RESULTS), exist_ok=True)
    with open(RESULTS, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")


def load_json(path):
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    if isinstance(d, dict):                                   # tolerate the object format
        d = [{"id": k, **(v if isinstance(v, dict) else {"document": v})} for k, v in d.items()]
    return d


def autocast_ctx(device, amp):
    if device == "cuda" and amp is not None:
        return torch.autocast(device_type="cuda", dtype=amp)
    return contextlib.nullcontext()


# =====================================================================================================
# Model surgery
# =====================================================================================================
class SplitMHA(nn.Module):
    """Drop-in replacement for nn.MultiheadAttention(batch_first=True) with SEPARATE q/k/v/o Linear layers
    (nn.MultiheadAttention packs q/k/v into one parameter and bypasses out_proj.forward, so no PEFT method
    could attach to it).  Weights are copied from the Phase-1 module, so the function is unchanged."""

    def __init__(self, mha):
        super().__init__()
        H = mha.embed_dim
        self.h, self.hd, self.p = mha.num_heads, H // mha.num_heads, mha.dropout
        dev = mha.in_proj_weight.device
        self.q_proj, self.k_proj, self.v_proj, self.o_proj = [nn.Linear(H, H).to(dev) for _ in range(4)]
        w, b = mha.in_proj_weight.data, mha.in_proj_bias.data
        for i, lin in enumerate((self.q_proj, self.k_proj, self.v_proj)):
            lin.weight.data.copy_(w[i * H:(i + 1) * H]); lin.bias.data.copy_(b[i * H:(i + 1) * H])
        self.o_proj.weight.data.copy_(mha.out_proj.weight.data); self.o_proj.bias.data.copy_(mha.out_proj.bias.data)

    def forward(self, query, key, value, key_padding_mask=None, attn_mask=None, need_weights=False):
        B, Tq, H = query.shape
        Tk = key.shape[1]
        q = self.q_proj(query).view(B, Tq, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(key).view(B, Tk, self.h, self.hd).transpose(1, 2)
        v = self.v_proj(value).view(B, Tk, self.h, self.hd).transpose(1, 2)
        mask = None
        if attn_mask is not None:                              # nn.MultiheadAttention semantics: bool True = BLOCKED
            if attn_mask.dtype == torch.bool:
                mask = torch.zeros(attn_mask.shape, dtype=q.dtype, device=q.device).masked_fill(attn_mask, float("-inf"))
            else:
                mask = attn_mask.to(q.dtype)
        if key_padding_mask is not None:                       # True = pad (ignore)
            kp = torch.zeros(B, 1, 1, Tk, dtype=q.dtype, device=q.device).masked_fill(key_padding_mask[:, None, None, :], float("-inf"))
            mask = kp if mask is None else mask + kp
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=self.p if self.training else 0.0)
        return self.o_proj(out.transpose(1, 2).reshape(B, Tq, H)), None


def convert_mha(model):
    for parent in list(model.modules()):
        for n, c in list(parent.named_children()):
            if isinstance(c, nn.MultiheadAttention):
                setattr(parent, n, SplitMHA(c))


def extend_ctx(model, cfg, new_ctx):
    """RoPE models have no position table (only the generate() crop uses max_position_embeddings); enc_dec has a
    learned table -> grow it (old rows kept, new rows tiled from them; it is fine-tuned in every experiment)."""
    old = cfg.max_position_embeddings
    if new_ctx <= old:
        return
    cfg.max_position_embeddings = new_ctx
    if hasattr(model, "pos_emb"):
        w = model.pos_emb.weight.data
        new = nn.Embedding(new_ctx, w.shape[1])
        new.weight.data[:old] = w
        new.weight.data[old:] = w[torch.arange(old, new_ctx) % old]
        model.pos_emb = new


def prune_depth(model, cfg, arch):
    """Student initialisation for distillation: keep every 2nd layer (indices 1,3,5,...) of the Phase-1 model."""
    keep = lambda lst: nn.ModuleList([l for i, l in enumerate(lst) if i % 2 == 1] or [lst[-1]])
    if arch == "enc_dec":
        model.encoder, model.decoder = keep(model.encoder), keep(model.decoder)
        cfg.num_layers = max(2, cfg.num_layers // 2)
    else:
        model.blocks = keep(model.blocks)
        cfg.num_layers = len(model.blocks)
        if arch == "efficient":                                  # old global layers 3,7 -> new 1,3
            cfg.global_every = max(1, cfg.global_every // 2)


def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr


# =====================================================================================================
# PEFT methods (from scratch)
# =====================================================================================================
class LoRALinear(nn.Module):
    """LoRA:  W' = W0 + (alpha/r) B A   (additive low-rank update)."""

    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale

    @torch.no_grad()
    def merged(self):
        self.base.weight += (self.B @ self.A) * self.scale
        return self.base


class LoRMALinear(nn.Module):
    """LoRMA (Low-Rank MULTIPLICATIVE Adaptation):  W' = M W0   with M = I_inflate(B A), M in R^{out x out}.

      mode="pi"   rank inflation by cyclic row permutation (row i of BA rotated by i). Initialisation from the paper:
                  B[:,0] = 1, B[:,1:] ~ small random, A[0,0] = 1, rest of A = 0  ->  I_pi(BA) = I, so training starts
                  exactly at the Phase-1 weights.
      mode="plus" additive inflation  M = I + (alpha/r) B A  (A random, B = 0 -> M = I at init).
    The pure product (BA) W0 has rank <= r, which is why an inflation operator is needed to keep W' full rank."""

    def __init__(self, base, r, mode="pi", alpha=None):
        super().__init__()
        out = base.out_features
        self.base, self.mode, self.out = base, mode, out
        self.A = nn.Parameter(torch.zeros(r, out))
        self.B = nn.Parameter(torch.zeros(out, r))
        if mode == "pi":
            with torch.no_grad():
                self.B[:, 0] = 1.0
                self.A[0, 0] = 1.0
                if r > 1:
                    self.B[:, 1:].normal_(0.0, 0.02)
            i = torch.arange(out)
            self.register_buffer("idx", (i[None, :] - i[:, None]) % out, persistent=False)   # M[i,j] = BA[i,(j-i)%out]
            self.scale = 1.0
        else:
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            self.scale = (alpha or r) / r

    def matrix(self):
        BA = self.B @ self.A
        if self.mode == "pi":
            return torch.gather(BA, 1, self.idx)
        return torch.eye(self.out, device=BA.device, dtype=BA.dtype) + self.scale * BA

    def weight(self):
        with torch.autocast(device_type=self.A.device.type, enabled=False):      # keep M @ W0 in fp32
            return self.matrix().float() @ self.base.weight.float()

    def forward(self, x):
        return F.linear(x, self.weight(), self.base.bias)

    @torch.no_grad()
    def merged(self):
        self.base.weight.copy_(self.weight())
        return self.base


class AdapterWrap(nn.Module):
    """Bottleneck adapter applied to the OUTPUT of `inner`:  y + Up(GELU(Down(y))),  Up initialised to 0."""

    def __init__(self, inner, dim, bottleneck):
        super().__init__()
        self.inner = inner
        self.down, self.up = nn.Linear(dim, bottleneck), nn.Linear(bottleneck, dim)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, *a, **k):
        y = self.inner(*a, **k)
        return y + self.up(F.gelu(self.down(y)))


def _wrap_children(model, match, factory):
    for parent in list(model.modules()):
        for n, c in list(parent.named_children()):
            if match(n, c):
                setattr(parent, n, factory(c))


def apply_technique(model, arch, tech, hp):
    """tech in: full | kd | lora | lorma_pi | lorma_plus | houlsby | pfeiffer | prompt"""
    model.prompt_len = 0
    if tech in ("full", "kd"):
        model.requires_grad_(True)
        return
    model.requires_grad_(False)
    targets = set(hp.get("targets", ATTN_T + FFN_T))
    r = hp.get("rank", 16)
    if tech == "lora":
        _wrap_children(model, lambda n, c: isinstance(c, nn.Linear) and n in targets,
                       lambda c: LoRALinear(c, r, hp.get("alpha", 2 * r), hp.get("dropout", 0.05)))
    elif tech in ("lorma_pi", "lorma_plus"):
        _wrap_children(model, lambda n, c: isinstance(c, nn.Linear) and n in targets,
                       lambda c: LoRMALinear(c, r, "pi" if tech == "lorma_pi" else "plus", hp.get("alpha")))
    elif tech in ("houlsby", "pfeiffer"):
        m = hp.get("bottleneck", 64)
        _wrap_children(model, lambda n, c: n == "ffn",
                       lambda c: AdapterWrap(c, c.w3.out_features, m))
        if tech == "houlsby":                                    # extra adapter after every attention sub-layer
            _wrap_children(model, lambda n, c: n == "o_proj" and isinstance(c, nn.Linear),
                           lambda c: AdapterWrap(c, c.out_features, m))
    elif tech == "prompt":
        P = hp.get("prompt_len", 50)
        emb = model.tok_emb.weight.data
        model.prompt_tuning = nn.Parameter(emb[torch.randint(0, min(5000, emb.size(0)), (P,))].clone())
        model.prompt_len = P
        first = model.encoder[0] if hasattr(model, "encoder") else model.blocks[0]

        def hook(module, args):
            x = args[0]
            pr = model.prompt_tuning.unsqueeze(0).expand(x.size(0), -1, -1).to(x.dtype)
            return (torch.cat([pr, x[:, P:]], dim=1),) + tuple(args[1:])
        first.register_forward_pre_hook(hook)
    else:
        raise ValueError(tech)
    if hasattr(model, "pos_emb"):                                # enc_dec: the (possibly extended) position table adapts
        model.pos_emb.weight.requires_grad_(True)


def merge_peft(model):
    """Fold LoRA / LoRMA into the base weights -> zero extra inference cost (adapters cannot be merged)."""
    for parent in list(model.modules()):
        for n, c in list(parent.named_children()):
            if isinstance(c, (LoRALinear, LoRMALinear)):
                setattr(parent, n, c.merged())


def trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}


# =====================================================================================================
# Experiment (de)serialisation
# =====================================================================================================
def load_phase1(path):
    model, cfg, _ = load_checkpoint(path, "cpu")
    return model, cfg


def build_experiment_model(meta):
    """Deterministic rebuild: Phase-1 weights -> (prune) -> (extend ctx) -> split MHA -> PEFT wrappers."""
    model, cfg = load_phase1(meta["phase1_ckpt"])
    if meta.get("scratch"):                                       # control experiment only: random init, same arch
        model = build_model(cfg)
    if meta.get("prune"):
        prune_depth(model, cfg, meta["arch"])
    extend_ctx(model, cfg, meta["ctx"])
    convert_mha(model)
    apply_technique(model, meta["arch"], meta["tech"], meta["hp"])
    return model


def exp_path(name):
    return f"{EXP_DIR}/{name}.pt"


def save_exp(name, meta, state):
    os.makedirs(EXP_DIR, exist_ok=True)
    torch.save({"meta": meta, "state": state}, exp_path(name))


def load_exp(name, device, merge=True):
    ck = torch.load(exp_path(name), map_location="cpu", weights_only=False)
    meta = ck["meta"]
    model = build_experiment_model(meta)
    missing, unexpected = model.load_state_dict(ck["state"], strict=False)
    assert not unexpected, f"unexpected keys when loading {name}: {unexpected[:5]}"
    if merge:
        merge_peft(model)
    return model.to(device).eval(), meta


def selftest_equivalence(ckpt_path, arch):
    """Max |logit difference| between the Phase-1 model and its SplitMHA-converted copy (must be ~1e-5)."""
    model, cfg = load_phase1(ckpt_path)
    conv = copy.deepcopy(model)
    convert_mha(conv)
    model.eval(); conv.eval()
    torch.manual_seed(0)
    with torch.no_grad():
        if arch == "enc_dec":
            src, tgt = torch.randint(4, 1000, (2, 90)), torch.randint(4, 1000, (2, 40))
            m = torch.zeros(2, 90, dtype=torch.bool); m[1, 70:] = True
            a, b = model(src, tgt, src_padding_mask=m)[0], conv(src, tgt, src_padding_mask=m)[0]
        else:
            ids = torch.randint(4, 1000, (2, 300))
            a, b = model(ids)[0], conv(ids)[0]
    return float((a - b).abs().max())


# =====================================================================================================
# Data -> token budgets, long-document selection, examples, batches
# =====================================================================================================
class Prep:
    def __init__(self, tok, arch, ctx, max_sum, prompt_len=0):
        self.tok, self.arch, self.fam = tok, arch, family(arch)
        self.ctx, self.max_sum, self.P = ctx, max_sum, prompt_len
        self.sep = tok.encode(SEP_TEXT, add_special_tokens=False).ids
        if self.fam == "causal":      # [P placeholders][bos] doc [sep] summary [eos]  must fit in ctx
            self.budget = ctx - max_sum - 1 - len(self.sep) - 1 - prompt_len - 4
        else:                         # encoder reads [P placeholders] doc ; decoder reads [bos] summary [eos]
            self.budget = min(ctx, ENC_SRC_MAX) - prompt_len
        assert self.budget > 64, f"context {ctx} too small for max_sum={max_sum}"

    def enc(self, text):
        return self.tok.encode(text, add_special_tokens=False).ids

    def doc_ids(self, doc, strategy, budget=None):
        budget = budget or self.budget
        if strategy == "trunc":
            return self.enc(doc)[:budget]
        if strategy == "headtail":
            ids = self.enc(doc)
            if len(ids) <= budget:
                return ids
            h = int(0.6 * budget)
            return ids[:h] + ids[-(budget - h):]
        if strategy == "select":
            sents = T.split_sentences(doc)
            lens = [len(e.ids) for e in self.tok.encode_batch(sents, add_special_tokens=False)]
            keep = T.select_sentences(sents, budget - 8, lens)
            return self.enc(" ".join(sents[i] for i in keep))[:budget]
        raise ValueError(strategy)

    def build_example(self, doc_ids, sum_ids=None):
        if self.fam == "causal":
            prefix = [UNK] * self.P + [BOS] + list(doc_ids) + self.sep
            if sum_ids is None:
                return prefix
            return {"ids": prefix + list(sum_ids) + [EOS], "p": len(prefix), "ls": len(sum_ids) + 1}
        src = [UNK] * self.P + list(doc_ids)
        if sum_ids is None:
            return src
        return {"src": src, "tgt": [BOS] + list(sum_ids) + [EOS], "ls": len(sum_ids) + 1}


def _pad(seqs, device):
    n = max(len(s) for s in seqs)
    out = torch.full((len(seqs), n), PAD, dtype=torch.long)
    for i, s in enumerate(seqs):
        out[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    return out.to(device)


def collate(exs, fam, device):
    ls = torch.tensor([e["ls"] for e in exs], device=device)
    if fam == "causal":
        return {"ids": _pad([e["ids"] for e in exs], device), "p": torch.tensor([e["p"] for e in exs], device=device), "ls": ls}
    return {"src": _pad([e["src"] for e in exs], device), "tgt": _pad([e["tgt"] for e in exs], device), "ls": ls}


def forward_pass(model, fam, b):
    """-> logits (B,Ls,V), targets (B,Ls), mask (B,Ls): ONLY the summary tokens (+eos), whatever the architecture."""
    if fam == "causal":
        logits = model(b["ids"])[0]
        B, Tn, V = logits.shape
        Ls = int(b["ls"].max())
        ar = torch.arange(Ls, device=logits.device)[None, :]
        lidx = (b["p"][:, None] - 1 + ar).clamp(max=Tn - 1)
        tidx = (b["p"][:, None] + ar).clamp(max=Tn - 1)
        lg = logits.gather(1, lidx.unsqueeze(-1).expand(-1, -1, V))
        tg = b["ids"].gather(1, tidx)
    else:
        src, tgt = b["src"], b["tgt"]
        lg = model(src, tgt[:, :-1], src_padding_mask=(src == PAD))[0]
        tg = tgt[:, 1:]
        ar = torch.arange(tg.size(1), device=tg.device)[None, :]
    return lg, tg, ar < b["ls"][:, None]


def ce_loss(lg, tg, mk):
    l = F.cross_entropy(lg.float().reshape(-1, lg.size(-1)), tg.reshape(-1), reduction="none").view_as(tg)
    return (l * mk).sum() / mk.sum().clamp(min=1)


def kd_loss(s_lg, t_lg, mk, temp):
    ls = F.log_softmax(s_lg.float() / temp, dim=-1)
    pt = F.softmax(t_lg.float() / temp, dim=-1)
    kl = (pt * (torch.log(pt.clamp_min(1e-9)) - ls)).sum(-1)
    return (kl * mk).sum() / mk.sum().clamp(min=1) * temp * temp


# =====================================================================================================
# Decoding: beam search with repetition control, works for every Phase-1 architecture
# =====================================================================================================
class LastHead(nn.Module):
    """Inference only: compute vocabulary logits for the LAST position only (saves ~T x V work per step)."""

    def __init__(self, head):
        super().__init__()
        self.head = head

    def forward(self, x):
        return self.head(x[:, -1:, :])


class Gen:
    def __init__(self, model, prep, device, amp):
        self.m, self.prep, self.dev, self.amp = model.eval(), prep, device, amp
        self.src_ids = None            # distinct token ids of the (selected) source text, used by the optional copy bias
        if not isinstance(model.lm_head, LastHead):
            model.lm_head = LastHead(model.lm_head)

    @torch.no_grad()
    def start(self, doc_ids):
        ex = self.prep.build_example(doc_ids)
        self.src_ids = torch.unique(torch.tensor(list(doc_ids), dtype=torch.long, device=self.dev)) if len(doc_ids) else None
        if self.prep.fam == "causal":
            self.prefix = ex
        else:
            with autocast_ctx(self.dev, self.amp):
                self.memory = self.m.encode(torch.tensor([ex], device=self.dev))

    @torch.no_grad()
    def logits(self, gens):
        with autocast_ctx(self.dev, self.amp):
            if self.prep.fam == "causal":
                ids = torch.tensor([self.prefix + g for g in gens], device=self.dev)
                lg = self.m(ids)[0][:, -1]
            else:
                tgt = torch.tensor([[BOS] + g for g in gens], device=self.dev)
                x = self.m._embed(tgt)
                Tn = tgt.size(1)
                cm = torch.triu(torch.full((Tn, Tn), float("-inf"), device=self.dev), diagonal=1)
                mem = self.memory.expand(len(gens), -1, -1)
                for layer in self.m.decoder:
                    x = layer(x, mem, cm, mem_key_padding_mask=None)
                lg = self.m.lm_head(self.m.norm_f(x))[:, -1]
        return lg.float()


def banned_ngram_tokens(g, n):
    if n <= 0 or len(g) < n - 1:
        return []
    prefix = tuple(g[-(n - 1):]) if n > 1 else ()
    out = set()
    for i in range(len(g) - n + 1):
        if tuple(g[i:i + n - 1]) == prefix:
            out.add(g[i + n - 1])
    return list(out)


@torch.no_grad()
def beam_search(gen, max_new, min_new=0, beams=4, len_pen=1.0, rep_pen=1.0, no_rep=4, copy_bias=0.0):
    live, done = [(0.0, [])], []
    for t in range(max_new):
        lg = gen.logits([g for _, g in live])
        nb, V = lg.shape
        lg[:, [PAD, BOS, UNK]] = float("-inf")
        if t < min_new:
            lg[:, EOS] = float("-inf")
        for b, (_, g) in enumerate(live):
            if g and rep_pen != 1.0:
                idx = torch.tensor(sorted(set(g)), device=lg.device)
                v = lg[b, idx]
                lg[b, idx] = torch.where(v < 0, v * rep_pen, v / rep_pen)
            ban = banned_ngram_tokens(g, no_rep)
            if ban:
                lg[b, ban] = float("-inf")
        if copy_bias and gen.src_ids is not None:      # soft copy prior: tokens that occur in the source get a logit bonus (-inf stays -inf)
            lg[:, gen.src_ids] += copy_bias
        sc = torch.tensor([s for s, _ in live], device=lg.device)[:, None] + torch.log_softmax(lg, dim=-1)
        top = torch.topk(sc.view(-1), min(2 * beams, nb * V))
        new = []
        for s, i in zip(top.values.tolist(), top.indices.tolist()):
            if s == float("-inf"):
                continue
            b, tok = divmod(i, V)
            g = live[b][1]
            if tok == EOS:
                done.append((s / ((len(g) + 1) ** len_pen), g))
            else:
                new.append((s, g + [tok]))
            if len(new) == beams:
                break
        if len(done) >= beams or not new:
            break
        live = new
    if not done:
        done = [(s / (max(1, len(g)) ** len_pen), g) for s, g in live]
    return max(done, key=lambda x: x[0])[1]


class Summarizer:
    """Wraps a fine-tuned model + a long-document strategy:  trunc | headtail | select | hier."""

    def __init__(self, model, prep, device, amp, chunk_new=150, overlap=1):
        self.g, self.prep, self.chunk_new, self.overlap = Gen(model, prep, device, amp), prep, chunk_new, overlap
        self.last_chunks = []

    def run(self, doc_ids, gen):
        self.g.start(doc_ids)
        out = beam_search(self.g, max_new=gen["max_new"], min_new=gen.get("min_new", 0), beams=gen.get("beams", 4),
                          len_pen=gen.get("len_pen", 1.0), rep_pen=gen.get("rep_pen", 1.0), no_rep=gen.get("no_rep", 4),
                          copy_bias=gen.get("copy_bias", 0.0))
        return self.prep.tok.decode(out, skip_special_tokens=True).strip()

    def summarize(self, doc, strategy, gen):
        self.last_chunks = []
        if strategy == "hier":
            return self.hier(doc, gen)
        return self.run(self.prep.doc_ids(doc, strategy), gen)

    def hier(self, doc, gen, max_levels=3):
        """Hierarchical: split into sentence-aligned overlapping chunks -> summarise every chunk -> concatenate
        (in document order) -> repeat while the result still exceeds the context -> final document-level summary."""
        P, text = self.prep, doc
        cgen = dict(gen, max_new=min(gen["max_new"], self.chunk_new), min_new=10, beams=max(1, gen.get("beams", 4) // 2))
        for _ in range(max_levels):
            sents = T.split_sentences(text)
            lens = [len(e.ids) for e in P.tok.encode_batch(sents, add_special_tokens=False)]
            if sum(lens) <= P.budget:
                break
            parts = []
            for i, j in T.chunk_sentences(lens, P.budget - 8, self.overlap):
                parts.append(self.run(P.enc(" ".join(sents[i:j]))[:P.budget], cgen))
            self.last_chunks.extend(parts)
            text = " ".join(p for p in parts if p)
        return self.run(P.doc_ids(text, "select"), gen)


# =====================================================================================================
# Metrics
# =====================================================================================================
def nanmean(xs):
    xs = [x for x in xs if x == x]
    return sum(xs) / len(xs) if xs else float("nan")


def compute_metrics(preds, refs, bert=True, bert_model=None):
    from rouge_score import rouge_scorer
    sc = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    r1 = rl = 0.0
    for p, r in zip(preds, refs):
        s = sc.score(r, p)
        r1 += s["rouge1"].fmeasure; rl += s["rougeL"].fmeasure
    n = max(1, len(preds))
    out = {"rouge1_f1": r1 / n, "rougeL_f1": rl / n, "bertscore_f1": float("nan")}
    if bert:
        try:
            from bert_score import score as bs
            _, _, F1 = bs([p if p.strip() else "." for p in preds], refs, lang="en", model_type=bert_model,
                          batch_size=16, verbose=False, device="cuda" if torch.cuda.is_available() else "cpu")
            out["bertscore_f1"] = float(F1.mean())
        except Exception as e:                                    # keep going: ROUGE is still reported
            print("[WARN] BERTScore failed:", type(e).__name__, e)
    return out


def analysis_means(preds, refs, srcs):
    rows = [T.analyse(p, r, s) for p, r, s in zip(preds, refs, srcs)]
    return {k: nanmean([r[k] for r in rows]) for k in rows[0]} if rows else {}


def eval_docs(summ, docs, strategy, gen, bert=True, bert_model=None):
    preds, chunks, t0 = [], [], time.time()
    for d in docs:
        preds.append(summ.summarize(d["document"], strategy, gen))
        chunks.append(list(summ.last_chunks))
    sec = (time.time() - t0) / max(1, len(docs))
    refs = [d["summary"] for d in docs]
    res = {**compute_metrics(preds, refs, bert, bert_model),
           **analysis_means(preds, refs, [d["document"] for d in docs]), "sec_per_doc": sec}
    return preds, chunks, res

In [ ]:
%%writefile run_exp.py
"""
Train + validate ONE Phase-6 experiment (one Phase-1 architecture x one adaptation technique), then save it.

  python run_exp.py --name lorma_decoder_only --arch decoder_only --tech lorma_pi --phase1_ckpt ... --lr 5e-5
  python run_exp.py --name kd_student --arch chunk_recurrent --tech kd --teacher <name of a finished experiment>
"""
import argparse
import json
import math
import os
import pickle
import random
import time

import torch
from tokenizers import Tokenizer

import p6lib as L


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", required=True)
    ap.add_argument("--arch", required=True, choices=["decoder_only", "enc_dec", "efficient", "chunk_recurrent"])
    ap.add_argument("--tech", required=True, choices=["full", "kd", "lora", "lorma_pi", "lorma_plus", "houlsby", "pfeiffer", "prompt"])
    ap.add_argument("--group", default="main", help="main | extra | control | kd")
    ap.add_argument("--phase1_ckpt", required=True)
    ap.add_argument("--tokenizer", default="artifacts/tokenizer.json")
    ap.add_argument("--train_file", required=True)
    ap.add_argument("--split_file", default="artifacts/p6_split.json")
    ap.add_argument("--ctx", type=int, default=1024)
    ap.add_argument("--max_sum", type=int, default=300, help="max summary tokens (train targets)")
    ap.add_argument("--strategy", default="select", help="long-document input strategy used for training: trunc|headtail|select")
    ap.add_argument("--eval_strategy", default=None)
    ap.add_argument("--steps", type=int, default=1200)
    ap.add_argument("--bs", type=int, default=4)
    ap.add_argument("--accum", type=int, default=4)
    ap.add_argument("--lr", type=float, default=5e-4)
    ap.add_argument("--wd", type=float, default=0.0)
    ap.add_argument("--warmup", type=int, default=60)
    ap.add_argument("--eval_every", type=int, default=200)
    ap.add_argument("--val_loss_n", type=int, default=48)
    ap.add_argument("--eval_n", type=int, default=40, help="val docs generated + scored (BERTScore/ROUGE) at the end")
    ap.add_argument("--max_train_docs", type=int, default=0)
    ap.add_argument("--rank", type=int, default=16)
    ap.add_argument("--alpha", type=float, default=32)
    ap.add_argument("--bottleneck", type=int, default=64)
    ap.add_argument("--prompt_len", type=int, default=50)
    ap.add_argument("--dropout", type=float, default=0.05)
    ap.add_argument("--scratch", action="store_true", help="CONTROL: random init instead of the Phase-1 weights")
    ap.add_argument("--teacher", default=None)
    ap.add_argument("--kd_alpha", type=float, default=0.5, help="weight of hard-label CE (rest = KL to teacher)")
    ap.add_argument("--kd_temp", type=float, default=2.0)
    ap.add_argument("--gen", default='{"max_new": 200, "min_new": 40, "beams": 4, "len_pen": 1.0, "rep_pen": 1.1, "no_rep": 4}')
    ap.add_argument("--no_bert", action="store_true")
    ap.add_argument("--bert_model", default=None)
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()

    L.set_seed(a.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    amp = L.get_amp_dtype()
    tok = Tokenizer.from_file(a.tokenizer)
    docs = {str(d["id"]): d for d in L.load_json(a.train_file)}
    split = json.load(open(a.split_file))
    train_ids = split["train"][: a.max_train_docs or None]
    val_ids = split["val"]

    hp = dict(rank=a.rank, alpha=a.alpha, bottleneck=a.bottleneck, prompt_len=a.prompt_len, dropout=a.dropout)
    meta = dict(name=a.name, arch=a.arch, tech=a.tech, group=a.group, hp=hp, ctx=a.ctx, max_sum=a.max_sum,
                prune=(a.tech == "kd"), scratch=a.scratch, phase1_ckpt=a.phase1_ckpt, train_strategy=a.strategy)
    model = L.build_experiment_model(meta).to(dev)
    n_total, n_train = L.count_params(model)
    prep = L.Prep(tok, a.arch, a.ctx, a.max_sum, hp["prompt_len"] if a.tech == "prompt" else 0)
    print(f"[{a.name}] {a.arch}/{a.tech}: total {n_total/1e6:.1f}M params, trainable {n_train/1e6:.3f}M "
          f"({100*n_train/n_total:.2f}%), doc budget {prep.budget} tokens, family {prep.fam}", flush=True)

    teacher, tprep = None, None
    if a.tech == "kd" and a.teacher:                              # (no --teacher = 'pruned student, hard labels only' control)
        teacher, tmeta = L.load_exp(a.teacher, dev, merge=True)
        teacher.requires_grad_(False)
        tprep = L.Prep(tok, tmeta["arch"], tmeta["ctx"], a.max_sum, tmeta["hp"]["prompt_len"] if tmeta["tech"] == "prompt" else 0)
        t_total, _ = L.count_params(teacher)
        assert t_total > n_total, f"teacher ({t_total/1e6:.1f}M) must be larger than the student ({n_total/1e6:.1f}M)"
        print(f"  teacher {a.teacher}: {tmeta['arch']}/{tmeta['tech']} {t_total/1e6:.1f}M params -> "
              f"student {n_total/1e6:.1f}M ({100*n_total/t_total:.0f}% of teacher)", flush=True)

    # ---- tokenised (document-input, summary) pieces; cached because 'select' is the slow part --------------------
    bp = max(prep.budget, tprep.budget if tprep else 0)
    os.makedirs("artifacts/cache", exist_ok=True)
    cpath = f"artifacts/cache/pieces_{a.strategy}_{bp}_{a.max_sum}_{len(train_ids)}.pkl"
    if os.path.exists(cpath):
        pieces = pickle.load(open(cpath, "rb"))
    else:
        t0 = time.time(); pieces = []
        for i in train_ids:
            d = docs[i]
            pieces.append({"doc": prep.doc_ids(d["document"], a.strategy, bp), "sum": prep.enc(d["summary"])[: a.max_sum]})
        pickle.dump(pieces, open(cpath, "wb"))
        print(f"  tokenised {len(pieces)} training docs in {time.time()-t0:.0f}s", flush=True)
    vpieces = [{"doc": prep.doc_ids(docs[i]["document"], a.strategy, bp), "sum": prep.enc(docs[i]["summary"])[: a.max_sum]}
               for i in val_ids[: a.val_loss_n]]

    def make_batch(idxs, pcs, pr):
        return L.collate([pr.build_example(pcs[i]["doc"][: pr.budget], pcs[i]["sum"]) for i in idxs], pr.fam, dev)

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=a.lr, weight_decay=a.wd)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / a.warmup) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, s / a.steps)))))
    use_scaler = (dev == "cuda" and amp == torch.float16)
    scaler = torch.cuda.amp.GradScaler(enabled=use_scaler)

    @torch.no_grad()
    def val_loss():
        model.eval(); tot = n = 0.0
        for i in range(0, len(vpieces), a.bs):
            b = make_batch(range(i, min(i + a.bs, len(vpieces))), vpieces, prep)
            with L.autocast_ctx(dev, amp):
                lg, tg, mk = L.forward_pass(model, prep.fam, b)
            tot += float(L.ce_loss(lg, tg, mk)) * float(mk.sum()); n += float(mk.sum())
        model.train()
        return tot / max(n, 1.0)

    rng = random.Random(a.seed)
    order, ptr = [], 0

    def next_idx():
        nonlocal order, ptr
        if ptr + a.bs > len(order):
            order = list(range(len(pieces))); rng.shuffle(order); ptr = 0
        out = order[ptr:ptr + a.bs]; ptr += a.bs
        return out

    if dev == "cuda":
        torch.cuda.reset_peak_memory_stats()
    model.train()
    best, best_state, hist, t_start = float("inf"), None, [], time.time()
    for step in range(1, a.steps + 1):
        opt.zero_grad(set_to_none=True)
        run_loss = 0.0
        for _ in range(a.accum):
            idx = next_idx()
            with L.autocast_ctx(dev, amp):
                lg, tg, mk = L.forward_pass(model, prep.fam, make_batch(idx, pieces, prep))
                loss = L.ce_loss(lg, tg, mk)
                if teacher is not None:
                    with torch.no_grad():
                        tl, ttg, tmk = L.forward_pass(teacher, tprep.fam, make_batch(idx, pieces, tprep))
                    assert tl.shape == lg.shape, (tl.shape, lg.shape)
                    loss = a.kd_alpha * loss + (1 - a.kd_alpha) * L.kd_loss(lg, tl, mk, a.kd_temp)
            scaler.scale(loss / a.accum).backward()
            run_loss += float(loss) / a.accum
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        if step % 25 == 0 or step == 1:
            print(f"  step {step}/{a.steps} loss {run_loss:.4f} lr {sched.get_last_lr()[0]:.2e} "
                  f"{(time.time()-t_start)/60:.1f} min", flush=True)
        if step % a.eval_every == 0 or step == a.steps:
            vl = val_loss(); hist.append((step, run_loss, vl))
            print(f"  >> step {step} val_loss {vl:.4f} (ppl {math.exp(min(vl, 20)):.1f})", flush=True)
            if vl < best:
                best, best_state = vl, L.trainable_state(model)
    train_min = (time.time() - t_start) / 60
    peak_gb = torch.cuda.max_memory_allocated() / 1e9 if dev == "cuda" else float("nan")

    model.load_state_dict(best_state, strict=False)
    L.save_exp(a.name, meta, best_state)
    file_mb = os.path.getsize(L.exp_path(a.name)) / 1e6
    del teacher, opt
    L.merge_peft(model)                                           # LoRA/LoRMA: zero inference overhead

    # ---- generation-based validation (BERTScore / ROUGE + redundancy / coverage numbers) ----------------------
    gen = json.loads(a.gen)
    strategy = a.eval_strategy or a.strategy
    docs_eval = [docs[i] for i in val_ids[: a.eval_n]]
    summ = L.Summarizer(model.to(dev), prep, dev, amp)
    t0 = time.time()
    preds, chunks, res = L.eval_docs(summ, docs_eval, strategy, gen, bert=not a.no_bert, bert_model=a.bert_model)
    os.makedirs("artifacts/val_preds", exist_ok=True)
    json.dump([{"id": d["id"], "summary": p, "reference": d["summary"]} for d, p in zip(docs_eval, preds)],
              open(f"artifacts/val_preds/{a.name}.json", "w"), ensure_ascii=False, indent=1)
    row = dict(kind="exp", tag=a.name, name=a.name, arch=a.arch, tech=a.tech, group=a.group, scratch=a.scratch,
               teacher=a.teacher, params_M=n_total / 1e6, trainable_M=n_train / 1e6, trainable_pct=100 * n_train / n_total,
               ckpt_file_MB=file_mb, train_min=train_min, peak_train_mem_GB=peak_gb, steps=a.steps, lr=a.lr,
               best_val_loss=best, best_val_ppl=math.exp(min(best, 20)), train_strategy=a.strategy, eval_strategy=strategy,
               eval_n=len(docs_eval), ctx=a.ctx, **res)
    L.append_result(row)
    print(f"[{a.name}] DONE  BERTScore {res['bertscore_f1']:.4f}  R1 {res['rouge1_f1']:.4f}  RL {res['rougeL_f1']:.4f}  "
          f"val_loss {best:.4f}  {res['sec_per_doc']:.1f}s/doc  train {train_min:.1f} min", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile infer.py
"""
Generate summaries with a saved experiment.
  --split val  : score on held-out validation docs (BERTScore/ROUGE + analysis metrics), logged to results.jsonl (kind='eval')
  --split test : write the competition file, list format  [{"id": ..., "summary": ...}]  (one entry per test document)
"""
import argparse
import json
import os
import time

import torch
from tokenizers import Tokenizer

import p6lib as L


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--exp", required=True)
    ap.add_argument("--split", choices=["val", "test"], required=True)
    ap.add_argument("--strategy", default="select", help="trunc | headtail | select | hier")
    ap.add_argument("--gen", default='{"max_new": 200, "min_new": 40, "beams": 4, "len_pen": 1.0, "rep_pen": 1.1, "no_rep": 4}')
    ap.add_argument("--n", type=int, default=40)
    ap.add_argument("--tag", default=None)
    ap.add_argument("--out", default=None)
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--tokenizer", default="artifacts/tokenizer.json")
    ap.add_argument("--train_file", default=None)
    ap.add_argument("--test_file", default=None)
    ap.add_argument("--split_file", default="artifacts/p6_split.json")
    ap.add_argument("--no_bert", action="store_true")
    ap.add_argument("--bert_model", default=None)
    a = ap.parse_args()

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    amp = L.get_amp_dtype()
    gen = json.loads(a.gen)
    tok = Tokenizer.from_file(a.tokenizer)
    model, meta = L.load_exp(a.exp, dev, merge=True)
    prep = L.Prep(tok, meta["arch"], meta["ctx"], meta["max_sum"], meta["hp"]["prompt_len"] if meta["tech"] == "prompt" else 0)
    summ = L.Summarizer(model, prep, dev, amp)
    print(f"loaded {a.exp} ({meta['arch']}/{meta['tech']}), strategy={a.strategy}, gen={gen}", flush=True)

    if a.split == "val":
        docs = {str(d["id"]): d for d in L.load_json(a.train_file)}
        ids = json.load(open(a.split_file))["val"][: a.n]
        dd = [docs[i] for i in ids]
        preds, chunks, res = L.eval_docs(summ, dd, a.strategy, gen, bert=not a.no_bert, bert_model=a.bert_model)
        tag = a.tag or f"{a.exp}__{a.strategy}"
        os.makedirs("artifacts/val_preds", exist_ok=True)
        json.dump([{"id": d["id"], "summary": p, "reference": d["summary"], "chunk_summaries": c}
                   for d, p, c in zip(dd, preds, chunks)], open(f"artifacts/val_preds/{tag}.json", "w"), ensure_ascii=False, indent=1)
        L.append_result(dict(kind="eval", tag=tag, name=a.exp, arch=meta["arch"], tech=meta["tech"], strategy=a.strategy,
                             eval_n=len(dd), **{f"gen_{k}": v for k, v in gen.items()}, **res))
        print(f"[{tag}] BERTScore {res['bertscore_f1']:.4f}  R1 {res['rouge1_f1']:.4f}  RL {res['rougeL_f1']:.4f}  "
              f"words {res['words']:.0f}  rep3 {res['rep3']:.3f}  {res['sec_per_doc']:.1f}s/doc", flush=True)
        return

    items = L.load_json(a.test_file)[: a.limit or None]
    out = a.out or f"artifacts/submission_{a.exp}.json"
    done = {}
    if a.resume and os.path.exists(out):
        done = {str(r["id"]): r["summary"] for r in json.load(open(out))}
        print(f"resuming with {len(done)} finished documents", flush=True)

    def dump():
        rows = [{"id": it["id"], "summary": done[str(it["id"])]} for it in items if str(it["id"]) in done]
        json.dump(rows, open(out + ".tmp", "w"), ensure_ascii=False, indent=1); os.replace(out + ".tmp", out)

    t0, todo = time.time(), [it for it in items if str(it["id"]) not in done]
    for k, it in enumerate(todo, 1):
        done[str(it["id"])] = summ.summarize(it["document"], a.strategy, gen) or "The court passed an order in the matter."
        if k % 10 == 0:
            el = time.time() - t0
            print(f"  {k}/{len(todo)}  {el/k:.1f}s/doc  ETA {el/k*(len(todo)-k)/60:.0f} min", flush=True)
        if k % 25 == 0:
            dump()
    dump()
    rows = json.load(open(out))
    assert len(rows) == len(items) and {str(r["id"]) for r in rows} == {str(i["id"]) for i in items}
    print(f"wrote {out}: {len(rows)} summaries, median {sorted(len(r['summary'].split()) for r in rows)[len(rows)//2]} words")


if __name__ == "__main__":
    main()

## 3. Helpers, data statistics and the hold-out split

In [ ]:
import subprocess, pandas as pd, numpy as np
from IPython.display import display
import matplotlib.pyplot as plt

def run(cmd):
    """Run a command, stream its output, STOP THE NOTEBOOK if it fails (SIGKILL = almost certainly out of memory)."""
    args = shlex.split(cmd)
    if args[0] == "python":
        args[0] = sys.executable
    print("$", " ".join(args)[:300], flush=True)
    t0 = time.time()
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=WORK)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    print(f"[exit {rc} after {(time.time()-t0)/60:.1f} min]", flush=True)
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}" + ("  <- SIGKILL: out of memory?" if rc == -9 else "") + f"\n{cmd[:300]}")

def load_results():
    p = "artifacts/results.jsonl"
    rows = [json.loads(l) for l in open(p)] if os.path.exists(p) else []
    df = pd.DataFrame(rows)
    if not len(df):
        return df
    df = df.drop_duplicates(subset=["kind", "tag"], keep="last")        # a re-run replaces the old row
    return df[~df.tag.str.startswith("smoke")].reset_index(drop=True)

def exp_table():
    df = load_results()
    return df[df.kind == "exp"].copy() if len(df) else df

def score_col(df):
    return "bertscore_f1" if "bertscore_f1" in df and df.bertscore_f1.notna().any() else "rougeL_f1"

MAIN_COLS = ["tag", "arch", "tech", "group", "params_M", "trainable_pct", "best_val_loss", "bertscore_f1", "rouge1_f1", "rougeL_f1",
             "words", "sec_per_doc", "train_min", "peak_train_mem_GB"]
def show(df, cols=MAIN_COLS, sort=True):
    if df is None or not len(df):
        print("(no results yet)"); return
    d = df[[c for c in cols if c in df.columns]]
    if sort and score_col(df) in d:
        d = d.sort_values(score_col(df), ascending=False)
    display(d.round(4).reset_index(drop=True))

import p6lib as L, p6text as T
from tokenizers import Tokenizer
tok = Tokenizer.from_file(TOKENIZER)
print("special ids:", {t: tok.token_to_id(t) for t in ["<pad>", "<bos>", "<eos>", "<unk>"]}, "(p6lib assumes 0,1,2,3)")
assert [tok.token_to_id(t) for t in ["<pad>", "<bos>", "<eos>", "<unk>"]] == [0, 1, 2, 3]

In [ ]:
docs_all  = L.load_json(TRAIN_FILE)
test_docs = L.load_json(TEST_FILE)
assert all("document" in d and "summary" in d and "id" in d for d in docs_all[:50]), "train.json items must have id/document/summary"
assert len({str(d["id"]) for d in docs_all}) == len(docs_all), "duplicate ids in train.json"
print(f"{len(docs_all)} training documents, {len(test_docs)} test documents")

rng = random.Random(SEED)
sample = rng.sample(docs_all, min(400, len(docs_all)))
enc_len = lambda t: len(tok.encode(t, add_special_tokens=False).ids)
dl = np.array([enc_len(d["document"]) for d in sample]); sl = np.array([enc_len(d["summary"]) for d in sample])
dw = np.array([len(d["document"].split()) for d in sample]); sw = np.array([len(d["summary"].split()) for d in sample])
q = lambda a: dict(zip(["p5", "p25", "median", "p75", "p95", "max"], np.percentile(a, [5, 25, 50, 75, 95, 100]).round(0)))
display(pd.DataFrame({"document tokens": q(dl), "document words": q(dw), "summary tokens": q(sl), "summary words": q(sw),
                      "compression (doc/sum tokens)": q(dl / np.maximum(sl, 1))}).T)

MAX_SUM = int(min(400, max(64, np.percentile(sl, 95))))           # summary tokens kept as training target / max generation
MED_SUM = int(np.median(sl))
# decoder-style models share one context window between document and summary (longer summary = shorter document), so they keep MAX_SUM;
# enc_dec has a separate decoder, so it can learn / write much longer summaries (references: median 773 tokens)
MAX_SUM_ARCH = {a: MAX_SUM for a in ARCHS}
if ENC_DEC_MAX_SUM:
    MAX_SUM_ARCH["enc_dec"] = max(MAX_SUM, int(ENC_DEC_MAX_SUM))
print("summary token cap per architecture:", MAX_SUM_ARCH)
GEN = dict(max_new=int(min(MAX_SUM, 1.2 * MED_SUM)), min_new=int(0.4 * MED_SUM), beams=4, len_pen=1.0, rep_pen=1.1, no_rep=4)
bud = {a: L.Prep(tok, a, CTX, MAX_SUM).budget for a in ("decoder_only", "enc_dec")}
print(f"MAX_SUM={MAX_SUM} tokens, median summary={MED_SUM} tokens -> default decoding {GEN}")
print(f"document budget per model: decoder-style {bud['decoder_only']} tokens, enc_dec {bud['enc_dec']} tokens")
print(f"documents longer than the budget: decoder-style {(dl > bud['decoder_only']).mean():.0%}, enc_dec {(dl > bud['enc_dec']).mean():.0%}",
      "-> long-document handling matters" if (dl > bud['decoder_only']).mean() > 0.2 else "-> most documents fit in one window")

SPLIT_FILE = "artifacts/p6_split.json"
if os.path.exists(SPLIT_FILE) and not FORCE:
    split = json.load(open(SPLIT_FILE)); print("re-using", SPLIT_FILE)
else:
    split = T.make_split([str(d["id"]) for d in docs_all], min(S["val_n"], max(2, len(docs_all) // 5)), SEED)
    json.dump(split, open(SPLIT_FILE, "w"))
by_id = {str(d["id"]): d for d in docs_all}
val_docs = [by_id[i] for i in split["val"]]
print(f"train {len(split['train'])} | held-out validation {len(split['val'])} (never used for training; model selection only)")

## 4. Long documents: how much of the reference summary is even *visible* to the model?

Model-free diagnostic for the "is legal information lost by chunking/truncation?" question. For each held-out document we compute the **recall of the reference summary's content words and of its "legal facts"** (numbers, statute/section references, capitalised names) **inside the input window** each strategy hands to the model. The *full document* column is the ceiling (what a perfect chunk-everything pipeline could see); the gap to it is information the model can never use. The histogram shows **where in the document** the reference-summary facts first appear.

In [ ]:
P_ = L.Prep(tok, "decoder_only", CTX, MAX_SUM)
rows = []
for d in val_docs[:min(120, len(val_docs))]:
    ref, doc = d["summary"], d["document"]
    rf, rc = T.facts(ref), T.content_set(ref)
    r = {"fits_in_window": enc_len(doc) <= P_.budget}
    for s in ["trunc", "headtail", "select"]:
        w = tok.decode(P_.doc_ids(doc, s))
        r[f"{s}|facts"], r[f"{s}|content"] = T.recall(rf, T.facts(w)), T.recall(rc, T.content_set(w))
    r["full document (ceiling)|facts"], r["full document (ceiling)|content"] = T.recall(rf, T.facts(doc)), T.recall(rc, T.content_set(doc))
    rows.append(r)
cov = pd.DataFrame(rows)
tab = pd.DataFrame({s: {"ref facts visible": cov[f"{s}|facts"].mean(), "ref content words visible": cov[f"{s}|content"].mean()}
                    for s in ["trunc", "headtail", "select", "full document (ceiling)"]}).T
display(tab.round(3))
print(f"{(~cov.fits_in_window).mean():.0%} of documents do not fit in one {P_.budget}-token window")
long_ = cov[~cov.fits_in_window]
if len(long_):
    print("restricted to the documents that do NOT fit:"); display(pd.DataFrame({s: {"ref facts visible": long_[f"{s}|facts"].mean(), "ref content words visible": long_[f"{s}|content"].mean()}
          for s in ["trunc", "headtail", "select", "full document (ceiling)"]}).T.round(3))

pos = []
for d in val_docs[:min(120, len(val_docs))]:
    low = d["document"].lower(); n = max(1, len(low))
    for f in T.facts(d["summary"]):
        k = low.find(f[2:])
        if k >= 0: pos.append(k / n)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
tab.plot.bar(ax=ax[0], rot=20); ax[0].set_title("Reference-summary information visible in the input window"); ax[0].set_ylim(0, 1)
ax[1].hist(pos, bins=20, color="#4c78a8"); ax[1].set_title("Where reference-summary facts first appear in the document"); ax[1].set_xlabel("relative position in document")
plt.tight_layout(); plt.show()
tab.to_csv("artifacts/longdoc_visibility.csv")

## 5. Smoke test (every code path, a few minutes)

1. `SplitMHA` conversion must leave every Phase-1 model's logits unchanged (**max |Δlogit| ≈ 1e-5**) — this is what makes LoRA / LoRMA / adapters on `enc_dec` and `chunk_recurrent` legitimate.
2. Each architecture trains 6 steps with **LoRMA**, each remaining technique (LoRA, Houlsby, Pfeiffer, prompt tuning, full) runs on `decoder_only`, distillation runs end-to-end, and `infer.py` writes a test file (`hier` strategy included).

If anything is broken (shape error, OOM, wrong path) it fails **here**, in minutes.

In [ ]:
TECH_HP = {   # learning rates are per technique (LoRMA multiplies W0 by a near-identity matrix, so it needs a much smaller lr)
    "full":       dict(lr=1e-4, wd=0.01),
    "lora":       dict(lr=5e-4, rank=16, alpha=32),
    "lorma_pi":   dict(lr=2e-4, rank=16),   # v1 used 5e-5: val_loss was still falling and LoRMA trailed LoRA/Pfeiffer on every architecture
    "lorma_plus": dict(lr=3e-4, rank=16, alpha=16),
    "houlsby":    dict(lr=1e-3, bottleneck=64),
    "pfeiffer":   dict(lr=1e-3, bottleneck=64),
    "prompt":     dict(lr=1e-2, prompt_len=50),
    "kd":         dict(lr=1e-4, wd=0.01, kd_alpha=0.5, kd_temp=2.0),
}
def elapsed_h():
    return ELAPSED_OFFSET_H + (time.time() - NB_START) / 3600
GRID_DEADLINE_H = SESSION_LIMIT_H - POST_GRID_RESERVE_H
def have_time(est_min):
    """True if one more experiment of ~est_min minutes can still finish before the grid deadline (always True in QUICK mode)."""
    return QUICK or elapsed_h() + est_min / 60 <= GRID_DEADLINE_H

BERT_FLAGS = ("" if USE_BERT else "--no_bert ") + (f"--bert_model {BERT_MODEL} " if BERT_MODEL else "")

def scale_gen(g, arch):
    """Length settings are written for MAX_SUM; an architecture trained on longer summaries gets them scaled proportionally."""
    f = MAX_SUM_ARCH.get(arch, MAX_SUM) / MAX_SUM
    return g if f == 1 else dict(g, max_new=int(round(g["max_new"] * f)), min_new=int(round(g.get("min_new", 0) * f)))

def arch_of(exp_name):
    return next((a for a in ARCHS if a in exp_name), "decoder_only")

def train_exp(name, arch, tech, group="main", teacher=None, scratch=False, strategy=None, smoke=False, **over):
    if os.path.exists(f"artifacts/exps/{name}.pt") and not FORCE and not smoke:
        print(f"[skip] {name} already trained (FORCE=True to redo)"); return
    hp = {**TECH_HP[tech], **over}
    sm = dict(steps=6, eval_every=6, warmup=2, bs=2, accum=1, max_train_docs=20, val_loss_n=4, eval_n=2) if smoke else {}
    B = {**S, **sm}
    flags = " ".join(f"--{k} {v}" for k, v in hp.items())
    gen = dict(GEN, max_new=24, min_new=4) if smoke else scale_gen(GEN, arch)
    cmd = (f"python run_exp.py --name {name} --arch {arch} --tech {tech} --group {group} --phase1_ckpt {shlex.quote(P1[arch])} "
           f"--tokenizer {shlex.quote(TOKENIZER)} --train_file {shlex.quote(TRAIN_FILE)} --split_file {SPLIT_FILE} --ctx {CTX} "
           f"--max_sum {MAX_SUM_ARCH[arch]} --strategy {strategy or TRAIN_STRATEGY} --steps {B['steps']} --bs {B['bs']} --accum {B['accum']} "
           f"--warmup {B['warmup']} --eval_every {B['eval_every']} --val_loss_n {B['val_loss_n']} --eval_n {B['eval_n']} "
           f"--max_train_docs {B['max_train_docs']} --gen {shlex.quote(json.dumps(gen))} --seed {SEED} {flags} "
           + BERT_FLAGS + ("--no_bert " if smoke and USE_BERT else "")
           + (f"--teacher {teacher} " if teacher else "") + ("--scratch " if scratch else ""))
    run(cmd)

def infer(exp, split="val", strategy="select", gen=None, n=None, tag=None, out=None, resume=False, limit=0, no_bert=False):
    gen = scale_gen(gen or GEN, arch_of(exp))
    cmd = (f"python infer.py --exp {exp} --split {split} --strategy {strategy} --gen {shlex.quote(json.dumps(gen))} "
           f"--tokenizer {shlex.quote(TOKENIZER)} --split_file {SPLIT_FILE} ")
    if split == "val":
        cmd += f"--train_file {shlex.quote(TRAIN_FILE)} --n {n or S['eval_n']} --tag {tag or exp + '__' + strategy} " + BERT_FLAGS + ("--no_bert " if no_bert else "")
    else:
        cmd += f"--test_file {shlex.quote(TEST_FILE)} --out {shlex.quote(out or f'artifacts/submission_{exp}.json')} " + ("--resume " if resume else "") + (f"--limit {limit} " if limit else "")
    run(cmd)
print("helpers ready")

In [ ]:
if SMOKE:
    print("1) SplitMHA equivalence on the real Phase-1 checkpoints")
    for arch in ARCHS:
        d = L.selftest_equivalence(P1[arch], arch)
        print(f"   {arch:16s} max |logit diff| = {d:.2e}", "OK" if d < 1e-3 else "<-- MISMATCH")
        assert d < 1e-3, f"SplitMHA changes the function of {arch}"
    print("2) tiny training runs (smoke_* are excluded from every results table)")
    for arch in ARCHS:
        train_exp(f"smoke_lorma_{arch}", arch, "lorma_pi", smoke=True)
    for tech in ["lora", "houlsby", "pfeiffer", "prompt", "full", "lorma_plus"]:
        train_exp(f"smoke_{tech}", "decoder_only", tech, smoke=True)
    train_exp("smoke_prompt_encdec", "enc_dec", "prompt", smoke=True)
    train_exp("smoke_lora_chunk", "chunk_recurrent", "lora", smoke=True)
    print("3) distillation (teacher = the 6-step decoder_only model; student = depth-pruned efficient)")
    train_exp("smoke_kd", "efficient", "kd", group="kd", teacher="smoke_full", smoke=True)
    train_exp("smoke_kd_enc", "enc_dec", "kd", group="kd", teacher="smoke_full", smoke=True)
    print("4) inference paths")
    for st in ["select", "hier"]:
        infer("smoke_lorma_decoder_only", "val", st, gen=dict(GEN, max_new=24, min_new=4), n=2, tag=f"smoke_val_{st}", no_bert=True)
    infer("smoke_lorma_enc_dec", "test", "select", gen=dict(GEN, max_new=24, min_new=4), limit=2, out="artifacts/smoke_submission.json")
    print(json.load(open("artifacts/smoke_submission.json"))[0])
    print("SMOKE TEST PASSED")

## 6. The experiment grid (§12.2)

| group | what | why |
|---|---|---|
| **main** | **LoRMA-π**, **LoRA**, **Pfeiffer adapters** on **each of the four Phase-1 architectures** (12 runs) | the required ≥ 3 techniques (LoRMA compulsory) on every Phase-1 model |
| **extra** | **full fine-tuning** on every architecture, then (behind the time guard) a LoRMA-π learning-rate bracket (`lr=6e-4`), Houlsby adapters, prompt tuning and LoRMA-+ on `decoder_only` | PEFT-vs-full question; LoRMA tuning; more techniques to compare |
| **control** | full fine-tuning from a **random** initialisation (`decoder_only`) | does Phase-1 pre-training help? (never submitted) |
| **kd** | §7 | mandatory distillation |

**Time guard:** the optional runs (everything after `main`, `full_*` and the control) are skipped once the session clock passes `SESSION_LIMIT_H - POST_GRID_RESERVE_H` (7 h by default), so the sweep and the test submissions can always finish inside Kaggle's 12 h limit.

`train_exp` skips finished runs, so you can stop and resume. If your quota is limited, set `RUN_EXTRA = False` first — the main group already satisfies the spec (the extras feed the analysis).
Hyper-parameter variations (e.g. a LoRMA learning-rate sweep) are allowed to *tune* a technique but do **not** count as separate techniques.

In [ ]:
RUN_MAIN, RUN_EXTRA, RUN_CONTROL = True, True, True
MAIN = [(f"{short}_{arch}", arch, tech, "main") for arch in ARCHS
        for short, tech in [("lorma", "lorma_pi"), ("lora", "lora"), ("pfeiffer", "pfeiffer")]]
EXTRA = [(f"full_{arch}", arch, "full", "extra") for arch in ARCHS]
EXTRA2 = [   # optional, run last, skipped by the time guard when the grid deadline is near; 5th element = hyper-parameter overrides
    ("lorma_decoder_only__lr6e-4", "decoder_only", "lorma_pi", "extra", dict(lr=6e-4)),   # LoRMA lr bracket: tuning, not a new technique
    ("houlsby_decoder_only", "decoder_only", "houlsby", "extra"),
    ("prompt_decoder_only", "decoder_only", "prompt", "extra"),
    ("lormaplus_decoder_only", "decoder_only", "lorma_plus", "extra")]
CONTROL = [("scratch_full_decoder_only", "decoder_only", "full", "control")]
print(len(MAIN), "main +", len(EXTRA) + len(EXTRA2), "extra +", len(CONTROL), "control experiments")

In [ ]:
if RUN_MAIN:
    for name, arch, tech, grp in MAIN:
        train_exp(name, arch, tech, grp)
show(exp_table()[lambda d: d.group == "main"] if len(exp_table()) else None)

In [ ]:
def run_guarded(items, scratch=False):
    for name, arch, tech, grp, *ov in items:
        if not (os.path.exists(f"artifacts/exps/{name}.pt") and not FORCE) and not have_time(25):
            print(f"[time guard] {elapsed_h():.1f} h used, grid deadline {GRID_DEADLINE_H:.1f} h -> skipping {name}")
            continue
        train_exp(name, arch, tech, grp, scratch=scratch, **(ov[0] if ov else {}))

if RUN_EXTRA:
    run_guarded(EXTRA)                      # full fine-tuning of every architecture (can be the best model of an architecture)
if RUN_CONTROL:
    run_guarded(CONTROL, scratch=True)      # Phase-1 init vs random init (needed for the report)
if RUN_EXTRA:
    run_guarded(EXTRA2)                     # optional extras, least important last
show(exp_table())


## 7. Mandatory knowledge distillation (§12.2.2)

* **Teacher** — the best fine-tuned experiment so far (largest BERTScore), required to have *more parameters* than the student.
* **Student** — a **depth-pruned copy of a Phase-1 model** (keeps layers 1, 3, 5, … so it still starts from Phase-1 weights) of `KD_STUDENT_ARCH`; it is then trained on the summaries with `α·CE + (1-α)·T²·KL(p_teacher ‖ p_student)` over the summary tokens (teacher and student may be different architectures: both emit logits over the same summary positions and share the tokenizer).
* **Control** `kd_ablate_*`: the *identical* pruned student trained with hard labels only, so the benefit of the teacher is isolated from the effect of pruning.

The comparison table reports parameters, memory, latency and quality of teacher → student.

In [ ]:
KD_STUDENT_ARCH = "efficient"      # any of the four; the student is a pruned copy of that Phase-1 model
KD_TEACHER = None                  # None = auto (best non-KD, non-control experiment that is larger than the student)

df = exp_table()
stu_meta = dict(arch=KD_STUDENT_ARCH, tech="kd", hp={}, ctx=CTX, prune=True, scratch=False, phase1_ckpt=P1[KD_STUDENT_ARCH])
stu_params = L.count_params(L.build_experiment_model(stu_meta))[0] / 1e6
cand = df[(df.tech != "kd") & (df.group != "control") & (df.params_M > stu_params)]
assert len(cand), "no experiment is larger than the pruned student -- train more experiments or use a smaller KD_STUDENT_ARCH"
teacher = KD_TEACHER or cand.sort_values(score_col(cand), ascending=False).iloc[0]["name"]
trow = df[df.name == teacher].iloc[0]
print(f"student {KD_STUDENT_ARCH} (pruned): {stu_params:.1f}M params | teacher {teacher}: {trow.arch}/{trow.tech}, {trow.params_M:.1f}M params "
      f"({stu_params/trow.params_M:.0%} of the teacher), {score_col(df)}={trow[score_col(df)]:.4f}")
train_exp(f"kd_{KD_STUDENT_ARCH}", KD_STUDENT_ARCH, "kd", group="kd", teacher=teacher)
train_exp(f"kd_ablate_{KD_STUDENT_ARCH}", KD_STUDENT_ARCH, "kd", group="control", kd_alpha=1.0)   # no teacher -> hard labels only
d = exp_table(); sc = score_col(d)
show(d[d.name.isin([teacher, f"kd_{KD_STUDENT_ARCH}", f"kd_ablate_{KD_STUDENT_ARCH}"])],
     MAIN_COLS + ["ckpt_file_MB"], sort=False)

## 8. Long-document strategies (§12.1) — on the best model so far

Same fine-tuned model, same decoding, four ways of handling a document that is longer than the context:

| strategy | idea |
|---|---|
| `trunc` | first *N* tokens |
| `headtail` | first 60 % + last 40 % of the window (facts at the start, holding at the end) |
| `select` | importance-based sentence selection (what the model was trained on) |
| `hier` | **hierarchical**: sentence-aligned **overlapping chunks → chunk summaries → (repeat) → document-level summary** |

`hier` is ~10× slower (many generations per document), so all four are compared on the same `hier_n` documents, and the three cheap ones additionally on `strat_n`. Because the model was *trained* on `select` inputs, `trunc`/`headtail` at inference are a train/test mismatch; the last cell therefore also **re-trains** the best configuration with `trunc` and `headtail` inputs for a fair comparison.

In [ ]:
d = exp_table(); sc = score_col(d)
BEST = d[d.group != "control"].sort_values(sc, ascending=False).iloc[0]
print("best model so far:", BEST["name"], f"({BEST.arch}/{BEST.tech}, {sc}={BEST[sc]:.4f})")
for st in ["trunc", "headtail", "select", "hier"]:
    infer(BEST["name"], "val", st, n=S["hier_n"], tag=f"strat_{BEST['name']}__{st}")
for st in ["trunc", "headtail", "select"]:
    infer(BEST["name"], "val", st, n=S["strat_n"], tag=f"stratfull_{BEST['name']}__{st}")
r = load_results(); r = r[r.kind == "eval"]
show(r[r.tag.str.startswith("strat_")], ["tag", "strategy", "eval_n", "bertscore_f1", "rouge1_f1", "rougeL_f1", "words", "rep3", "ref_fact_recall", "ref_content_recall", "fact_hallucination", "sec_per_doc"])
show(r[r.tag.str.startswith("stratfull_")], ["tag", "strategy", "eval_n", "bertscore_f1", "rouge1_f1", "rougeL_f1", "words", "rep3", "ref_fact_recall", "sec_per_doc"])

In [ ]:
# Hierarchical information loss: how many reference facts survive each stage?  (input window -> chunk summaries -> final summary)
hp_ = f"artifacts/val_preds/strat_{BEST['name']}__hier.json"
if os.path.exists(hp_):
    rows = []
    for it in json.load(open(hp_)):
        doc = by_id[str(it["id"])]["document"]; rf = T.facts(it["reference"])
        chunk_text = " ".join(it["chunk_summaries"]) if it["chunk_summaries"] else ""
        rows.append({"ref facts in document": T.recall(rf, T.facts(doc)),
                     "... in concatenated chunk summaries": T.recall(rf, T.facts(chunk_text)) if chunk_text else np.nan,
                     "... in final hierarchical summary": T.recall(rf, T.facts(it["summary"])),
                     "chunk-summary words": len(chunk_text.split()), "n chunk summaries": len(it["chunk_summaries"])})
    display(pd.DataFrame(rows).mean().round(3).to_frame("hierarchical pipeline (mean over docs)"))
    print("Compare 'in document' (ceiling) with the later stages: the drop is the legal information lost by chunk-then-summarise.")

In [ ]:
# Training-time long-document ablation: retrain the best configuration with trunc / headtail inputs
bn, ba, bt = BEST["name"], BEST["arch"], BEST["tech"]
for st in ["trunc", "headtail"]:
    train_exp(f"{bn}__train_{st}", ba, bt, group="extra", strategy=st)
d = exp_table()
show(d[d.name.isin([bn, f"{bn}__train_trunc", f"{bn}__train_headtail"])], ["tag", "train_strategy", "eval_strategy", "best_val_loss", "bertscore_f1", "rouge1_f1", "rougeL_f1", "words", "ref_fact_recall"], sort=False)

## 9. Generation length and decoding (§12.5: *how does generation length affect BERTScore and ROUGE?*)

Sweep of the target length (`max_new` / `min_new` as fractions of `MAX_SUM`, the longest summary the models were trained to write) and of the repetition controls (n-gram blocking, repetition penalty) on the best model, same documents for every setting. Stage 1 sweeps length and repetition control; stage 2 adds an optional **copy bias** (a logit bonus for tokens that occur in the selected source text) on top of the stage-1 winner. The best setting by BERTScore becomes `FINAL_GEN`.

*v1 bug fixed:* the old length sweep used fractions of the **median reference length** (773 tokens) capped at `MAX_SUM` (400), so four of its five "length" settings were identical and neither the plot nor `FINAL_GEN` could say anything about length.

In [ ]:
# lengths are fractions of MAX_SUM (= the longest summary the models were trained to write; enc_dec settings are scaled automatically)
rel = [(0.4, 0.30), (0.6, 0.45), (0.8, 0.60), (1.0, 0.77), (1.0, 0.95)]   # (max_new, min_new) / MAX_SUM; (1.0, 0.77) = the default GEN
SWEEP = [dict(GEN, max_new=int(a_ * MAX_SUM), min_new=int(b_ * MAX_SUM)) for a_, b_ in rel]
SWEEP += [dict(GEN, beams=1),                      # greedy (cost / quality trade-off)
          dict(GEN, no_rep=0, rep_pen=1.0),        # no repetition control at all
          dict(GEN, rep_pen=1.0),                  # n-gram blocking only: rep_pen=1.1 penalises ordinary function words ("the", "of")
          dict(GEN, rep_pen=1.05),
          dict(GEN, no_rep=3), dict(GEN, no_rep=6)]   # legal text repeats 4-grams legitimately ("of the Income Tax Act")
SWEEP_STRAT = FINAL_STRATEGY if "FINAL_STRATEGY" in globals() else "select"

def run_cfgs(cfgs, start):
    for i, g in enumerate(cfgs, start):
        infer(BEST["name"], "val", SWEEP_STRAT, gen=g, n=S["sweep_n"], tag=f"sweep_{BEST['name']}__{i}")

def sweep_table():
    r = load_results(); sw = r[r.tag.str.startswith(f"sweep_{BEST['name']}__")].copy()
    sw["idx"] = [int(t.split("__")[-1]) for t in sw.tag]
    sw = sw[sw.idx < len(SWEEP)].copy()
    sw["setting"] = [f"max_new={g['max_new']} min_new={g['min_new']} beams={g['beams']} no_rep={g['no_rep']} rep_pen={g['rep_pen']} "
                     f"len_pen={g['len_pen']} copy_bias={g.get('copy_bias', 0)}" for g in (SWEEP[i] for i in sw.idx)]
    return sw

# stage 1: length and repetition control
run_cfgs(SWEEP, 0)
sw = sweep_table(); sc = score_col(sw)
FINAL_GEN = SWEEP[int(sw.sort_values(sc, ascending=False).iloc[0].idx)]
print("stage-1 winner:", FINAL_GEN)

# stage 2: soft copy bias on top of the stage-1 winner (a logit bonus for tokens that occur in the selected source text; still beam-search
# generation from the fine-tuned model, it only makes the model reuse the document's names, sections and legal terms more readily)
STAGE2 = [dict(FINAL_GEN, copy_bias=cb) for cb in (1.0, 2.0, 3.0)]
run_cfgs(STAGE2, len(SWEEP)); SWEEP += STAGE2
sw = sweep_table(); sc = score_col(sw)
show(sw, ["setting", "bertscore_f1", "rouge1_f1", "rougeL_f1", "words", "rep3", "distinct2", "ref_fact_recall", "sec_per_doc"], sort=False)
FINAL_GEN = SWEEP[int(sw.sort_values(sc, ascending=False).iloc[0].idx)]
print("FINAL_GEN =", FINAL_GEN)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ls_ = sw[sw.idx < len(rel)].sort_values("words")
for m_ in ["bertscore_f1", "rouge1_f1", "rougeL_f1"]:
    ax[0].plot(ls_.words, ls_[m_], "o-", label=m_)
ax[0].set_xlabel("mean generated words"); ax[0].set_title("Quality vs generation length"); ax[0].legend()
ax[1].scatter(sw.rep3, sw[sc]); ax[1].set_xlabel("3-gram repetition rate"); ax[1].set_ylabel(sc); ax[1].set_title("Repetition vs quality (all sweep settings)")
plt.tight_layout(); plt.show()


## 10. Select the best four (different Phase-1 models) and write the submissions (§12.3.5)

For every Phase-1 architecture the best non-control experiment (by validation BERTScore F1) is chosen, giving four submissions on **four different Phase-1 models**. `FINAL_STRATEGY` is the best long-document strategy from §8 (`hier` only if it clearly wins, since it is ~10× slower); `FINAL_GEN` comes from §9. Set `PINNED = {"enc_dec": "<exp name>", ...}` to force a particular experiment for an architecture.

Output format = **list format** `[{"id": ..., "summary": ...}]`, exactly one entry per test document. The checks below verify this.

In [ ]:
PINNED = {}
d = exp_table(); sc = score_col(d)
cand = d[d.group != "control"]
FINAL = {}
for arch in ARCHS:
    c = cand[cand.arch == arch]
    FINAL[arch] = PINNED.get(arch) or c.sort_values(sc, ascending=False).iloc[0]["name"]
sel = d[d.name.isin(FINAL.values())]
show(sel, MAIN_COLS)

r = load_results(); r = r[(r.kind == "eval") & r.tag.str.startswith("strat_")]
if len(r):
    st = r.groupby("strategy")[score_col(r)].mean()
    print("long-document strategies (same documents):"); print(st.round(4).to_string())
    FINAL_STRATEGY = st.drop("hier", errors="ignore").idxmax()
    if "hier" in st and st["hier"] > st.drop("hier").max() + 0.005:
        FINAL_STRATEGY = "hier"
else:
    FINAL_STRATEGY = "select"
FINAL_GEN = globals().get("FINAL_GEN", GEN)
print("FINAL_STRATEGY =", FINAL_STRATEGY, "| FINAL_GEN =", FINAL_GEN)

techs = set(cand.tech)
print("\nspec check:",
      "| LoRMA run:", bool(techs & {"lorma_pi", "lorma_plus"}),
      "| KD run:", "kd" in techs,
      f"| distinct techniques: {len(techs)} {sorted(techs)}",
      "| submissions on 4 different Phase-1 models:", len({cand[cand.name == v].iloc[0].arch for v in FINAL.values()}) == 4)
print("techniques of the 4 submissions:", {a: cand[cand.name == n].iloc[0].tech for a, n in FINAL.items()})

In [ ]:
os.makedirs("artifacts/submissions", exist_ok=True)
for arch, name in FINAL.items():
    ts = d[d.name == name].iloc[0].train_strategy
    strat = FINAL_STRATEGY if ts == "select" else ts           # a model trained on trunc/headtail inputs is decoded the same way
    out = f"artifacts/submissions/{arch}__{name}.json"
    infer(name, "test", strat, gen=FINAL_GEN, out=out, resume=True)

In [ ]:
test_ids = [str(t["id"]) for t in test_docs]
for arch, name in FINAL.items():
    f = f"artifacts/submissions/{arch}__{name}.json"
    rows = json.load(open(f))
    assert isinstance(rows, list) and all(set(r) == {"id", "summary"} for r in rows), "must be list format [{id, summary}]"
    ids = [str(r["id"]) for r in rows]
    assert len(ids) == len(set(ids)) == len(test_ids) and set(ids) == set(test_ids), "need exactly one summary per test document"
    nw = pd.Series([len(r["summary"].split()) for r in rows])
    print(f"{arch:16s} {name:28s} OK  {len(rows)} summaries | words min/median/max {nw.min()}/{nw.median():.0f}/{nw.max()} | empty {int((nw == 0).sum())}")
BEST_FINAL = d[d.name.isin(FINAL.values())].sort_values(score_col(d), ascending=False).iloc[0]
shutil.copy(f"artifacts/submissions/{BEST_FINAL.arch}__{BEST_FINAL['name']}.json", "submission_best.json")
for arch, name in FINAL.items():
    shutil.copy(f"artifacts/submissions/{arch}__{name}.json", f"submission_{arch}.json")
print("\nCodabench files (working dir):", sorted(f for f in os.listdir('.') if f.startswith('submission_')))
print("best on validation:", BEST_FINAL["name"], "->", "submission_best.json")

## 11. Analysis (§12.5) — every table / plot below is computed from `artifacts/results.jsonl`

Validation numbers come from the held-out split (`eval_n` documents, generated with beam search); they are *model-selection* numbers — the Codabench score on the private references is the final judge. With ~50 validation documents differences below ≈ 0.005 BERTScore are within noise; say so in the report.

In [ ]:
# 11.1 Which techniques work?  (BERTScore F1, architecture x technique)
d = exp_table(); sc = score_col(d)
dd = d[(d.group != "control") & ~d.name.str.contains("__train_|__lr")]
display(dd.pivot_table(index="arch", columns="tech", values=sc).round(4))
show(dd.groupby("tech")[[sc, "rouge1_f1", "rougeL_f1", "trainable_pct", "train_min", "peak_train_mem_GB"]].mean().reset_index().rename(columns={"tech": "tag"}),
     ["tag", sc, "rouge1_f1", "rougeL_f1", "trainable_pct", "train_min", "peak_train_mem_GB"])
fig, ax = plt.subplots(figsize=(9, 3.6))
dd.pivot_table(index="arch", columns="tech", values=sc).plot.bar(ax=ax, rot=0); ax.set_ylabel(sc); ax.set_title("Adaptation technique x Phase-1 architecture")
ax.set_ylim(max(0, dd[sc].min() - 0.03), dd[sc].max() + 0.01); plt.tight_layout(); plt.show()

In [ ]:
# 11.2 Can parameter-efficient methods retain the quality of full fine-tuning?
full = d[(d.tech == "full") & (d.group != "control")].set_index("arch")
rows = []
for _, r in d[~d.tech.isin(["full", "kd"]) & (d.group != "control") & ~d.name.str.contains("__train_|__lr")].iterrows():
    if r.arch in full.index:
        f = full.loc[r.arch]
        rows.append({"experiment": r["name"], "arch": r.arch, "tech": r.tech, "trainable %": r.trainable_pct, sc: r[sc], "full FT " + sc: f[sc],
                     "% of full-FT quality": 100 * r[sc] / f[sc], "train time / full": r.train_min / f.train_min,
                     "peak mem / full": r.peak_train_mem_GB / f.peak_train_mem_GB, "stored MB": r.ckpt_file_MB, "full-FT stored MB": f.ckpt_file_MB})
pe = pd.DataFrame(rows); display(pe.sort_values("% of full-FT quality", ascending=False).round(3))

In [ ]:
# 11.3 Domain adaptation / usefulness of Phase-1 representations: Phase-1 initialisation vs random initialisation (same arch, same budget)
sc_ = d[d.scratch == True]
if len(sc_):
    a = sc_.iloc[0].arch; p1 = d[(d.arch == a) & (d.tech == "full") & (d.scratch != True) & (d.group != "control")]
    show(pd.concat([p1, sc_]), ["tag", "arch", "scratch", "best_val_loss", "best_val_ppl", sc, "rouge1_f1", "rougeL_f1", "ref_fact_recall"], sort=False)
    if len(p1):
        print(f"Phase-1 initialisation vs random: d{sc} = {p1.iloc[0][sc] - sc_.iloc[0][sc]:+.4f}, d val_loss = {p1.iloc[0].best_val_loss - sc_.iloc[0].best_val_loss:+.4f}")
# frozen-representation probe: PEFT methods that keep 99%+ of the Phase-1 weights frozen reaching a given fraction of full-FT quality
display(d[d.trainable_pct < 5][["tag", "trainable_pct", sc]].round(4).sort_values(sc, ascending=False).head(8))

In [ ]:
# 11.4 Knowledge distillation: teacher -> student quality vs size / memory / latency
kd = d[d.tech == "kd"]
if len(kd):
    k = kd[kd.group == "kd"].iloc[0]; t = d[d.name == k.teacher].iloc[0]
    ab = d[d.name == f"kd_ablate_{k.arch}"]
    rows = [dict(role="teacher", name=t["name"], params_M=t.params_M, **{sc: t[sc], "rougeL_f1": t.rougeL_f1, "sec_per_doc": t.sec_per_doc, "peak_train_mem_GB": t.peak_train_mem_GB})]
    rows.append(dict(role="student (KD)", name=k["name"], params_M=k.params_M, **{sc: k[sc], "rougeL_f1": k.rougeL_f1, "sec_per_doc": k.sec_per_doc, "peak_train_mem_GB": k.peak_train_mem_GB}))
    if len(ab):
        a_ = ab.iloc[0]; rows.append(dict(role="student, no teacher (control)", name=a_["name"], params_M=a_.params_M, **{sc: a_[sc], "rougeL_f1": a_.rougeL_f1, "sec_per_doc": a_.sec_per_doc, "peak_train_mem_GB": a_.peak_train_mem_GB}))
    kt = pd.DataFrame(rows); display(kt.round(4))
    print(f"student keeps {100*k[sc]/t[sc]:.1f}% of the teacher's {sc} with {100*k.params_M/t.params_M:.0f}% of the parameters and "
          f"{t.sec_per_doc/k.sec_per_doc:.2f}x faster generation" + (f"; the teacher improves the student by {k[sc]-ab.iloc[0][sc]:+.4f} {sc} over hard-label training" if len(ab) else ""))

In [ ]:
# 11.5 Size, memory and cost of every experiment
cost = d[["tag", "arch", "tech", "params_M", "trainable_M", "trainable_pct", "ckpt_file_MB", "peak_train_mem_GB", "train_min", "sec_per_doc", sc]].copy()
cost["fp16 model MB"] = cost.params_M * 2
display(cost.sort_values("trainable_pct").round(3).reset_index(drop=True))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].scatter(d.trainable_pct, d[sc]); ax[0].set_xscale("log"); ax[0].set_xlabel("trainable parameters (%)"); ax[0].set_ylabel(sc); ax[0].set_title("Quality vs trainable fraction")
ax[1].scatter(d.sec_per_doc, d[sc]); ax[1].set_xlabel("generation seconds / document"); ax[1].set_title("Quality vs inference cost")
for _, r in d.iterrows():
    ax[0].annotate(r.tech, (r.trainable_pct, r[sc]), fontsize=6)
plt.tight_layout(); plt.show()

In [ ]:
# 11.6 Factuality, coverage and redundancy of the generated summaries (validation docs)
cols = ["tag", "tech", sc, "words", "ref_fact_recall", "ref_content_recall", "fact_hallucination", "ext4", "rep3", "distinct2"]
show(d, cols)
print("ref_fact_recall: share of the reference's numbers / section refs / names that the summary contains (coverage of key facts)\n"
      "ref_content_recall: same for content words | fact_hallucination: share of the summary's facts NOT in the source document (lower = more faithful)\n"
      "ext4: share of summary 4-grams copied from the source | rep3 / distinct2: repetition inside the summary (lower rep3 = less redundant)")
fig, ax = plt.subplots(1, 3, figsize=(13, 3.3))
for a_, (x, t_) in zip(ax, [("rep3", "redundancy"), ("fact_hallucination", "hallucinated facts"), ("ref_fact_recall", "fact coverage")]):
    a_.scatter(d[x], d[sc]); a_.set_xlabel(x); a_.set_ylabel(sc); a_.set_title(t_)
plt.tight_layout(); plt.show()

In [ ]:
# 11.7 Qualitative check: reference vs the four submitted models on two validation documents
pv = {a: json.load(open(f"artifacts/val_preds/{n}.json")) for a, n in FINAL.items() if os.path.exists(f"artifacts/val_preds/{n}.json")}
for k_ in range(2):
    first = next(iter(pv.values()))[k_]
    print("=" * 110, f"\nDOCUMENT {first['id']} (first 500 chars): {by_id[str(first['id'])]['document'][:500]!r}\n\nREFERENCE:\n{first['reference'][:900]}")
    for a, rows in pv.items():
        print(f"\n--- {a} / {FINAL[a]}:\n{rows[k_]['summary'][:900]}")

In [ ]:
# 11.8 Everything in one table (feeds the report)
r_all = load_results()
r_all.to_csv("artifacts/results_summary_phase6.csv", index=False)
show(exp_table(), MAIN_COLS + ["ckpt_file_MB", "ref_fact_recall", "fact_hallucination", "rep3"])
print("saved artifacts/results_summary_phase6.csv")

## 12. Report checklist (§12.5) — where each answer comes from

- [ ] **Experiments:** ≥ 4 substantially different (LoRMA-π, LoRA, Pfeiffer, KD; plus full FT, Houlsby, prompt tuning, LoRMA-+), LoRMA present, KD present (§6, §7, check printed in §10). Explain why they are "substantially different": multiplicative vs additive low-rank update, bottleneck modules in the residual stream, input-space prompt, teacher-student transfer of output distributions.
- [ ] **All four Phase-1 models** used; best four submissions on four different models (§10 table). State which Phase-1 checkpoint (Tier-1/Tier-2, context length) each came from.
- [ ] **Which techniques are effective** — §11.1 pivot + means; discuss LoRMA's learning-rate sensitivity (multiplicative update of every output row): v1 used lr 5e-5 and trailed LoRA / Pfeiffer on every architecture; v2 uses 2e-4 and adds a 6e-4 bracket run (`lorma_decoder_only__lr6e-4`, excluded from the technique means).
- [ ] **PEFT vs full fine-tuning** — §11.2 (% of full-FT quality, trainable %, time, memory, stored MB).
- [ ] **Long-document strategies** — §4 (information visible per strategy), §8 (trunc / headtail / select / hierarchical, train-time ablation, information lost through chunk summaries).
- [ ] **Legal facts, reasoning & coverage** — §11.6 (`ref_fact_recall`, `ref_content_recall`, `fact_hallucination`) + §11.7 qualitative samples. Remember `facts()` is a proxy (numbers, section references, capitalised names), not a legal-correctness judge.
- [ ] **Redundancy / repetition** — §11.6 `rep3`, `distinct2`, and the decoding sweep in §9 (n-gram blocking, repetition penalty).
- [ ] **Generation length vs BERTScore / ROUGE** — §9 plot (ROUGE favours matching the reference length more strongly than BERTScore does).
- [ ] **Domain adaptation / Phase-1 representations** — §11.3 (Phase-1 init vs random init, and how much quality survives with > 95 % of Phase-1 weights frozen).
- [ ] **KD** — §11.4 (quality retained vs parameters, latency, memory; teacher vs no-teacher control).
- [ ] **Model size, memory, inference cost** — §11.5.
- [ ] **Limitations to state honestly:** small validation set (noise ≈ ±0.005); LoRMA formulation follows the paper's permutation-inflation description (`I_π`, identity initialisation) and an additive `I + BA` variant I implemented from its description; AdaLoRA, QLoRA and prefix tuning were not run; hierarchical summarisation reuses a model trained on whole-document → summary pairs (no chunk-level supervision).